In [4]:
!pip install -q openai requests sentence-transformers faiss-cpu pypdf gradio pandas

In [2]:
from getpass import getpass

NEXUS_API_KEY = getpass("Enter replacement API key: ").strip()

print("Correct format:", NEXUS_API_KEY.startswith("sk-"))

Enter replacement API key: ··········
Correct format: True


In [3]:
from openai import OpenAI

BASE_URL = "https://nexusapi.navigatelabs.ai"

client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url=BASE_URL
)

print("Nexus API client initialized successfully!")

Nexus API client initialized successfully!


In [5]:
models = client.models.list()

available_models = [
    model.id for model in models.data
]

print("AVAILABLE MODELS:")

for model in available_models:
    print(model)

AVAILABLE MODELS:
nano-banana
whisper-1
nova-micro
text-embedding-3-small
gpt-4o-mini-tts
gpt-4.1-nano


In [ ]:
MODEL_NAME = "nova-micro"

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": "Explain Artificial Intelligence in simple English."
            }
        ],
        max_tokens=150
    )

    print("Model:", MODEL_NAME)
    print("AI Response:")
    print(response.choices[0].message.content)

except Exception as e:
    print("Error:", type(e).__name__)
    print("Details:", str(e))

In [ ]:
from google.colab import files

print("Upload your government scheme PDF documents")

uploaded_files = files.upload()

print("\nUploaded files:")

for filename in uploaded_files:
    print(filename)

In [ ]:
from pypdf import PdfReader

documents = []

for filename in uploaded_files:
    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()

        if page_text and page_text.strip():
            documents.append({
                "text": page_text,
                "source": filename,
                "page": page_number
            })

print("PDF extraction completed!")
print("Total pages extracted:", len(documents))

In [ ]:
for doc in documents[:3]:
    print("\nPage:", doc["page"])
    print(doc["text"][:1000])
    print("-" * 50)

In [ ]:
def create_chunks(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:
    chunks = create_chunks(doc["text"])

    for chunk in chunks:
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })

print("Text chunking completed!")
print("Total chunks:", len(all_chunks))

In [ ]:
for i, chunk in enumerate(all_chunks[:3]):
    print(f"\nCHUNK {i+1}")
    print(chunk["text"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("-" * 50)

In [ ]:
test_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=["PM-KISAN provides financial assistance to eligible farmer families."]
)

print("Embedding test successful!")
print("Embedding dimensions:", len(test_embedding.data[0].embedding))

In [ ]:
import numpy as np

texts = [chunk["text"] for chunk in all_chunks]

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = np.array(
    [item.embedding for item in embedding_response.data],
    dtype="float32"
)

print("All embeddings generated successfully!")
print("Total embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

In [ ]:
import faiss

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

print("FAISS vector database created successfully!")
print("Total vectors stored:", index.ntotal)

In [ ]:
def retrieve_information(question, top_k=3):

    question_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [question_embedding.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, top_k)

    results = []

    for idx in indices[0]:
        if idx != -1:
            results.append(all_chunks[idx])

    return results


print("SchemeSense retrieval function created successfully!")

In [ ]:
question = "How much financial assistance does PM-KISAN provide?"

results = retrieve_information(question)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Information:", result["text"])
    print("-" * 60)

In [ ]:
def generate_answer(question):

    # Retrieve relevant information from PDF
    results = retrieve_information(question)

    # Prepare context with source details
    context = ""

    for result in results:
        context += (
            f"\nSource: {result['source']}, "
            f"Page: {result['page']}\n"
            f"Information: {result['text']}\n"
        )

    # Send retrieved information to AI model
    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": """
                You are SchemeSense, a government scheme information assistant.

                Answer using only the provided PDF information.
                Explain in simple English.
                Do not invent eligibility rules or benefits.
                If the answer is not available in the PDF,
                clearly say: Information not found in the provided document.

                Always mention the source filename and page number.
                Remember that a student-created sample document
                is not an official government notification.
                """
            },
            {
                "role": "user",
                "content": f"""
                Question: {question}

                Retrieved PDF information:
                {context}

                Give a clear answer with source details.
                """
            }
        ],
        max_tokens=500
    )

    return response.choices[0].message.content


print("SchemeSense RAG answer function created successfully!")

In [ ]:
def generate_answer(question):

    results = retrieve_information(question)

    context = ""

    for result in results:
        context += f"""
        Source: {result['source']}
        Page: {result['page']}
        Text: {result['text']}
        """

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": """
                You are SchemeSense, a government scheme assistant.

                Carefully read ALL retrieved text before answering.

                Rules:
                1. Search the retrieved text for the exact answer.
                2. Do not say information is missing if it is present.
                3. Answer only using retrieved information.
                4. Use simple English.
                5. Mention the source filename and page.
                6. If the information is genuinely absent, say:
                   Information not found in the provided document.
                """
            },
            {
                "role": "user",
                "content": f"""
                Question: {question}

                Retrieved information:
                {context}

                Answer the question using the retrieved information.
                """
            }
        ],
        max_tokens=400
    )

    return response.choices[0].message.content


print("Updated SchemeSense RAG function created!")

In [ ]:
question = "How much financial assistance does PM-KISAN provide?"

answer = generate_answer(question)

print(answer)

In [ ]:
def collect_citizen_details():

    print("===== SCHEMESENSE CITIZEN REGISTRATION =====")

    name = input("Enter your name: ")
    age = int(input("Enter your age: "))
    state = input("Enter your state: ")
    occupation = input("Enter your occupation: ")
    land_owner = input("Do you own agricultural land? (yes/no): ")
    annual_income = input("Enter your annual family income: ")

    citizen = {
        "name": name,
        "age": age,
        "state": state,
        "occupation": occupation,
        "land_owner": land_owner,
        "annual_income": annual_income
    }

    return citizen


print("Citizen interview module created successfully!")

In [ ]:
citizen = collect_citizen_details()

print("\n===== CITIZEN DETAILS =====")

for key, value in citizen.items():
    print(key, ":", value)

In [ ]:
def check_eligibility(citizen):

    question = f"""
    Check whether this citizen may qualify for PM-KISAN.

    Citizen details:
    {citizen}

    Explain:
    1. What eligibility information is available?
    2. What additional information is needed?
    3. Which official rules must be verified?

    Do not declare the citizen eligible or ineligible
    unless the provided document supports that conclusion.
    """

    answer = generate_answer(question)

    return answer


print("Eligibility assessment function created successfully!")

In [ ]:
eligibility_result = check_eligibility(citizen)

print("===== SCHEMESENSE ELIGIBILITY REPORT =====")
print(eligibility_result)

In [ ]:
import gradio as gr

def schemesense_chat(message, history):

    answer = generate_answer(message)

    return answer


demo = gr.ChatInterface(
    fn=schemesense_chat,
    title="SchemeSense – Government Scheme Assistant",
    description=(
        "Ask questions about government schemes. "
        "Answers are generated using the uploaded PDF documents."
    ),
    examples=[
        "What is PM-KISAN?",
        "How much financial assistance does PM-KISAN provide?",
        "How many instalments are provided?",
        "Who can benefit from PM-KISAN?"
    ]
)

print("SchemeSense chatbot interface created successfully!")

In [ ]:
demo.launch(share=True, debug=True)

In [ ]:
from google.colab import files
from pypdf import PdfReader

print("Upload additional official government scheme PDFs")

new_uploaded_files = files.upload()

for filename in new_uploaded_files:

    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text and page_text.strip():

            documents.append({
                "text": page_text,
                "source": filename,
                "page": page_number,
                "document_type": "official"
            })

print("\nAdditional PDFs uploaded successfully!")
print("Total extracted pages:", len(documents))

In [ ]:
all_chunks = []

for doc in documents:

    chunks = create_chunks(doc["text"])

    for chunk in chunks:

        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "document_type": doc.get("document_type", "sample")
        })

print("Updated text chunks created!")
print("Total chunks:", len(all_chunks))

In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = np.array(
    [item.embedding for item in embedding_response.data],
    dtype="float32"
)

print("Updated embeddings generated!")
print("Total embeddings:", len(embeddings))
print("Dimensions:", embeddings.shape[1])

In [ ]:
import faiss
import numpy as np

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Create a new FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])

# Store all embeddings
index.add(embeddings)

print("FAISS database updated successfully!")
print("Total vectors stored:", index.ntotal)

In [ ]:
question = "What government schemes are available in the uploaded documents?"

results = retrieve_information(question, top_k=5)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Information:", result["text"][:500])
    print("-" * 50)

In [ ]:
def get_available_schemes():

    schemes = {
        "PM-KISAN": [
            "SchemeSense_PM_KISAN_Sample_Reference.pdf",
            "RevisedPM-KISANOperationalGuidelines(English).pdf"
        ]
    }

    return schemes


available_schemes = get_available_schemes()

print("AVAILABLE GOVERNMENT SCHEMES")
print("=" * 40)

for scheme in available_schemes:
    print(scheme)

In [ ]:
def search_selected_scheme(question, selected_scheme):

    allowed_sources = available_schemes[selected_scheme]

    scheme_chunks = [
        chunk for chunk in all_chunks
        if chunk["source"] in allowed_sources
    ]

    if not scheme_chunks:
        return []

    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk["source"] in allowed_sources
    ]

    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [query_response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, index.ntotal)

    results = []

    for idx in indices[0]:
        if idx in selected_indices:
            results.append(all_chunks[idx])

        if len(results) == 3:
            break

    return results


print("Scheme-specific search created successfully!")

In [ ]:
selected_scheme = "PM-KISAN"

question = "What are the exclusion categories under PM-KISAN?"

results = search_selected_scheme(question, selected_scheme)

print("SELECTED SCHEME:", selected_scheme)
print("=" * 50)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Information:", result["text"][:600])
    print("-" * 50)

In [ ]:
def generate_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme
    )

    if not results:
        return "No relevant information found in the selected scheme documents."

    context = ""

    for result in results:
        context += f"""
        Source: {result['source']}
        Page: {result['page']}
        Information: {result['text']}
        """

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": """
                You are SchemeSense, a government scheme assistant.

                Answer only using the retrieved documents.
                Use simple English.
                Never invent government rules.
                Explain eligibility conditions carefully.
                Mention exact source filename and page number.
                If information is missing, clearly say so.
                Do not make a final eligibility decision
                without all required information.
                """
            },
            {
                "role": "user",
                "content": f"""
                Selected scheme: {selected_scheme}

                Citizen question: {question}

                Retrieved documents:
                {context}

                Give a clear answer with source citations.
                """
            }
        ],
        max_tokens=600
    )

    return response.choices[0].message.content


print("SchemeSense source-based answer system created!")

In [ ]:
answer = generate_scheme_answer(
    "Who is excluded from receiving PM-KISAN benefits?",
    "PM-KISAN"
)

print("===== SCHEMESENSE VERIFIED RESPONSE =====")
print(answer)

In [ ]:
import gradio as gr

def citizen_interview(
    age,
    occupation,
    owns_land,
    income_tax,
    government_employee,
    monthly_pension,
    professional
):

    details = f"""
    Applicant age: {age}
    Occupation: {occupation}
    Owns agricultural land: {owns_land}
    Paid income tax last assessment year: {income_tax}
    Government employee: {government_employee}
    Monthly pension: {monthly_pension}
    Practicing registered professional: {professional}
    """

    question = f"""
    Assess the following PM-KISAN applicant information.

    {details}

    Explain:
    1. Which exclusion conditions may apply?
    2. Which eligibility details are still missing?
    3. What should the applicant verify with official authorities?

    Do not declare final eligibility.
    Use only the uploaded official PM-KISAN guidelines.
    Include source filenames and page numbers.
    """

    return generate_scheme_answer(question, "PM-KISAN")


print("Citizen eligibility interview function created!")

In [ ]:
interview_app = gr.Interface(
    fn=citizen_interview,
    inputs=[
        gr.Number(label="Applicant Age", value=30),
        gr.Textbox(label="Occupation", placeholder="Example: Farmer"),
        gr.Radio(["Yes", "No"], label="Do you own agricultural land?"),
        gr.Radio(["Yes", "No"], label="Did you pay income tax last assessment year?"),
        gr.Radio(["Yes", "No"], label="Are you a government employee?"),
        gr.Number(label="Monthly pension (₹)", value=0),
        gr.Radio(["Yes", "No"], label="Are you a registered practicing professional?")
    ],
    outputs=gr.Markdown(label="SchemeSense Assessment"),
    title="SchemeSense – PM-KISAN Eligibility Interview",
    description="Answer these questions to receive a preliminary document-based assessment."
)

interview_app.launch(share=True)

In [ ]:

def improved_eligibility_interview(
    name,
    age,
    state,
    land_owner,
    land_record,
    family_members,
    institutional_land,
    constitutional_post,
    government_employee,
    monthly_pension,
    income_tax,
    professional,
    nri
):

    citizen = {
        "Name": name,
        "Age": age,
        "State": state,
        "Agricultural land ownership": land_owner,
        "Land record available": land_record,
        "Farmer family members": family_members,
        "Institutional landholding": institutional_land,
        "Constitutional or elected position": constitutional_post,
        "Government employment": government_employee,
        "Monthly pension": monthly_pension,
        "Paid income tax last assessment year": income_tax,
        "Registered practicing professional": professional,
        "NRI status": nri
    }

    return citizen


print("Improved PM-KISAN questionnaire created!")


In [ ]:

import gradio as gr

def assess_applicant(
    name, age, state, land_owner, land_record,
    family_members, institutional_land,
    constitutional_post, government_employee,
    monthly_pension, income_tax, professional, nri
):

    citizen = improved_eligibility_interview(
        name, age, state, land_owner, land_record,
        family_members, institutional_land,
        constitutional_post, government_employee,
        monthly_pension, income_tax, professional, nri
    )

    question = f"""
    Review this PM-KISAN applicant's details:

    {citizen}

    Compare the information with the retrieved official
    PM-KISAN operational guidelines.

    Provide:
    1. Possible exclusion conditions.
    2. Missing information.
    3. Conditions that need official verification.
    4. Relevant source filenames and page numbers.

    Do not make a final eligibility decision.
    Do not assume missing information is negative.
    """

    return generate_scheme_answer(question, "PM-KISAN")


eligibility_form = gr.Interface(
    fn=assess_applicant,
    inputs=[
        gr.Textbox(label="Applicant Name"),
        gr.Number(label="Age"),
        gr.Textbox(label="State"),
        gr.Radio(["Yes", "No", "Not sure"], label="Do you own agricultural land?"),
        gr.Radio(["Yes", "No", "Not sure"], label="Are land records available?"),
        gr.Textbox(label="Farmer family members (husband, wife and minor children)"),
        gr.Radio(["Yes", "No", "Not sure"], label="Is the land institutionally owned?"),
        gr.Radio(["Yes", "No", "Not sure"], label="Does any family member hold a listed constitutional or elected position?"),
        gr.Radio(["Yes", "No", "Not sure"], label="Does any family member hold a listed government or local-body position?"),
        gr.Number(label="Monthly pension (₹)", value=0),
        gr.Radio(["Yes", "No", "Not sure"], label="Did any family member pay income tax in the last assessment year?"),
        gr.Radio(["Yes", "No", "Not sure"], label="Is any family member a registered practicing professional?"),
        gr.Radio(["Yes", "No", "Not sure"], label="Is the farmer family NRI?"),
    ],
    outputs=gr.Markdown(label="Preliminary Assessment Report"),
    title="SchemeSense – PM-KISAN Eligibility Assessment",
    description="Provide applicant details for a preliminary document-based assessment."
)

eligibility_form.launch(share=True)


In [ ]:

def classify_pm_kisan(details):

    exclusion_reasons = []
    missing_details = []

    if details["Agricultural land ownership"] == "No":
        missing_details.append(
            "Confirm whether the farmer family meets the landholding requirement."
        )

    if details["Agricultural land ownership"] == "Not sure":
        missing_details.append("Confirm agricultural land ownership.")

    if details["Land record available"] != "Yes":
        missing_details.append("Verify land records.")

    if details["Institutional landholding"] == "Yes":
        exclusion_reasons.append("Institutional landholding.")

    if details["Constitutional or elected position"] == "Yes":
        exclusion_reasons.append(
            "Possible constitutional or elected-position exclusion."
        )

    if details["Government employment"] == "Yes":
        exclusion_reasons.append(
            "Check the applicable government employment exclusion and exceptions."
        )

    if details["Monthly pension"] >= 10000:
        exclusion_reasons.append(
            "Check the pension exclusion and applicable exceptions."
        )

    if details["Paid income tax last assessment year"] == "Yes":
        exclusion_reasons.append("Income-tax exclusion.")

    if details["Registered practicing professional"] == "Yes":
        exclusion_reasons.append("Registered practicing professional exclusion.")

    if details["NRI status"] == "Yes":
        exclusion_reasons.append("NRI farmer family exclusion.")

    if exclusion_reasons:
        status = "POSSIBLE EXCLUSION"

    elif missing_details:
        status = "MORE INFORMATION REQUIRED"

    else:
        status = "POTENTIALLY ELIGIBLE – OFFICIAL VERIFICATION REQUIRED"

    return {
        "status": status,
        "exclusion_reasons": exclusion_reasons,
        "missing_details": missing_details
    }


print("Eligibility classification function created!")


In [ ]:
sample_citizen = {
    "Agricultural land ownership": "Yes",
    "Land record available": "Yes",
    "Institutional landholding": "No",
    "Constitutional or elected position": "No",
    "Government employment": "No",
    "Monthly pension": 0,
    "Paid income tax last assessment year": "No",
    "Registered practicing professional": "No",
    "NRI status": "No"
}

report = classify_pm_kisan(sample_citizen)

print("===== SCHEMESENSE ELIGIBILITY REPORT =====")
print("Status:", report["status"])
print("Possible exclusions:", report["exclusion_reasons"])
print("Missing details:", report["missing_details"])

In [ ]:

def generate_eligibility_report(details):

    result = classify_pm_kisan(details)

    report = f"""
# SCHEMESENSE – ELIGIBILITY REPORT

## Applicant Information

**Applicant:** {details.get("Name", "Test Applicant")}

**State:** {details.get("State", "Not provided")}

## Assessment Result

### {result["status"]}

## Possible Exclusion Reasons

"""

    if result["exclusion_reasons"]:
        for reason in result["exclusion_reasons"]:
            report += f"- {reason}\n"
    else:
        report += "- No listed exclusion detected from the answers.\n"

    report += "\n## Missing Information\n"

    if result["missing_details"]:
        for item in result["missing_details"]:
            report += f"- {item}\n"
    else:
        report += "- No missing fields detected by this preliminary checklist.\n"

    report += """
## Important Notice

This is an AI-assisted preliminary screening report.
It is not an official government eligibility certificate.

Verify all conditions, exceptions and current rules
through the official PM-KISAN portal and authorities.
"""

    return report


print("Eligibility report generator created!")


In [ ]:
sample_citizen.update({
    "Name": "Sample Applicant",
    "State": "Andhra Pradesh"
})

final_report = generate_eligibility_report(sample_citizen)

from IPython.display import display, Markdown

display(Markdown(final_report))

In [ ]:

def final_assessment(
    name, age, state, land_owner, land_record,
    family_members, institutional_land,
    constitutional_post, government_employee,
    monthly_pension, income_tax, professional, nri
):

    details = {
        "Name": name,
        "Age": age,
        "State": state,
        "Agricultural land ownership": land_owner,
        "Land record available": land_record,
        "Farmer family members": family_members,
        "Institutional landholding": institutional_land,
        "Constitutional or elected position": constitutional_post,
        "Government employment": government_employee,
        "Monthly pension": monthly_pension,
        "Paid income tax last assessment year": income_tax,
        "Registered practicing professional": professional,
        "NRI status": nri
    }

    return generate_eligibility_report(details)


print("Final assessment function created successfully!")


In [ ]:

final_app = gr.Interface(
    fn=final_assessment,
    inputs=[
        gr.Textbox(label="Applicant Name"),
        gr.Number(label="Age"),
        gr.Textbox(label="State"),
        gr.Radio(["Yes", "No", "Not sure"], label="Agricultural land ownership"),
        gr.Radio(["Yes", "No", "Not sure"], label="Land records available"),
        gr.Textbox(label="Farmer family members"),
        gr.Radio(["Yes", "No", "Not sure"], label="Institutional landholding"),
        gr.Radio(["Yes", "No", "Not sure"], label="Constitutional or elected position"),
        gr.Radio(["Yes", "No", "Not sure"], label="Government employee"),
        gr.Number(label="Monthly pension (₹)", value=0),
        gr.Radio(["Yes", "No", "Not sure"], label="Paid income tax last assessment year"),
        gr.Radio(["Yes", "No", "Not sure"], label="Registered practicing professional"),
        gr.Radio(["Yes", "No", "Not sure"], label="NRI status")
    ],
    outputs=gr.Markdown(label="Eligibility Report"),
    title="SchemeSense – Automatic Eligibility Report",
    description="Complete the questionnaire to receive a preliminary PM-KISAN assessment."
)

final_app.launch(share=True)


In [ ]:

import gradio as gr

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SchemeSense AI Dashboard"
) as dashboard:

    gr.Markdown("""
    # 🏛️ SCHEMESENSE
    ### AI-Based Government Scheme Information & Eligibility Assistant

    Discover government schemes, ask questions and check
    preliminary eligibility using uploaded scheme documents.

    ---
    """)

    with gr.Tab("🏠 Home"):

        gr.Markdown("""
        ## Welcome to SchemeSense!

        SchemeSense uses Retrieval-Augmented Generation (RAG)
        to retrieve information from government scheme documents.

        ### Our Features

        - 📄 Government PDF document processing
        - 🤖 AI-powered scheme chatbot
        - 🔎 FAISS-based document retrieval
        - 📋 Citizen eligibility questionnaire
        - 📊 Preliminary assessment reports

        ### Available Scheme Documents

        - PM-KISAN Sample Reference
        - PM-KISAN Revised Operational Guidelines

        **Project:** SchemeSense – A Verified AI Agent
        for Citizen Scheme Eligibility

        **Technology:** Python, Google Colab, FAISS,
        RAG, Nova Micro API and Gradio.
        """)

    with gr.Tab("🤖 AI Chatbot"):

        gr.Markdown("""
        ## Ask SchemeSense

        Ask questions about the uploaded government scheme PDFs.
        """)

        gr.ChatInterface(
            fn=schemesense_chat,
            examples=[
                "What is PM-KISAN?",
                "How much financial assistance is provided?",
                "Who is excluded from PM-KISAN?",
                "How many instalments are provided?"
            ]
        )

    with gr.Tab("📋 Eligibility Checker"):

        gr.Markdown("""
        ## PM-KISAN Preliminary Eligibility Checker

        Enter the applicant details below.
        """)

        name = gr.Textbox(label="Applicant Name")
        age = gr.Number(label="Age")
        state = gr.Textbox(label="State")

        land_owner = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Agricultural land ownership"
        )

        land_record = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Land records available"
        )

        family_members = gr.Textbox(
            label="Farmer family members"
        )

        institutional_land = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Institutional landholding"
        )

        constitutional_post = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Constitutional or elected position"
        )

        government_employee = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Government employee"
        )

        monthly_pension = gr.Number(
            label="Monthly pension (₹)",
            value=0
        )

        income_tax = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Paid income tax last assessment year"
        )

        professional = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="Registered practicing professional"
        )

        nri = gr.Radio(
            ["Yes", "No", "Not sure"],
            label="NRI status"
        )

        assess_button = gr.Button(
            "Generate Assessment Report",
            variant="primary"
        )

        assessment_output = gr.Markdown(
            label="Preliminary Assessment Report"
        )

        assess_button.click(
            fn=final_assessment,
            inputs=[
                name, age, state, land_owner, land_record,
                family_members, institutional_land,
                constitutional_post, government_employee,
                monthly_pension, income_tax, professional, nri
            ],
            outputs=assessment_output
        )

    gr.Markdown("""
    ---
    **Disclaimer:** SchemeSense provides AI-assisted preliminary
    information. Final eligibility must be verified using current
    official government rules and authorities.
    """)


print("SchemeSense dashboard created successfully!")

dashboard.launch(share=True, debug=True)


In [ ]:

def classify_pm_kisan(details):

    exclusion_reasons = []
    missing_details = []

    # Check required personal information
    if not details.get("Name", "").strip():
        missing_details.append("Applicant name.")

    if not details.get("State", "").strip():
        missing_details.append("Applicant state.")

    if details.get("Age") is None:
        missing_details.append("Applicant age.")

    if not details.get("Farmer family members", "").strip():
        missing_details.append("Farmer family member details.")

    # Check Yes / No / Not sure answers
    fields = [
        "Agricultural land ownership",
        "Land record available",
        "Institutional landholding",
        "Constitutional or elected position",
        "Government employment",
        "Paid income tax last assessment year",
        "Registered practicing professional",
        "NRI status"
    ]

    for field in fields:
        answer = str(details.get(field, "")).strip().lower()

        if answer not in ["yes", "no"]:
            missing_details.append(
                f"Confirm: {field}."
            )

    # Agricultural land requirement
    if str(details.get("Agricultural land ownership", "")).lower() == "no":
        exclusion_reasons.append(
            "The applicant reports no agricultural land ownership. "
            "The basic landholding requirement needs verification."
        )

    # Other possible exclusion conditions
    if details.get("Institutional landholding") == "Yes":
        exclusion_reasons.append("Institutional landholding.")

    if details.get("Constitutional or elected position") == "Yes":
        exclusion_reasons.append(
            "Possible exclusion based on constitutional or elected position."
        )

    if details.get("Government employment") == "Yes":
        exclusion_reasons.append(
            "Government employment requires checking the applicable "
            "category and exceptions."
        )

    pension = details.get("Monthly pension")

    if pension is None:
        missing_details.append("Monthly pension amount.")
    elif pension >= 10000:
        exclusion_reasons.append(
            "Monthly pension of ₹10,000 or more requires checking "
            "the applicable exclusion and exceptions."
        )

    if details.get("Paid income tax last assessment year") == "Yes":
        exclusion_reasons.append("Income-tax exclusion.")

    if details.get("Registered practicing professional") == "Yes":
        exclusion_reasons.append(
            "Possible exclusion for registered practicing professionals."
        )

    if details.get("NRI status") == "Yes":
        exclusion_reasons.append("NRI farmer family exclusion.")

    # Final preliminary status
    if exclusion_reasons and missing_details:
        status = "POSSIBLE EXCLUSION – MORE INFORMATION REQUIRED"

    elif exclusion_reasons:
        status = "POSSIBLE EXCLUSION – OFFICIAL VERIFICATION REQUIRED"

    elif missing_details:
        status = "MORE INFORMATION REQUIRED"

    else:
        status = "NO LISTED EXCLUSION DETECTED – OFFICIAL VERIFICATION REQUIRED"

    return {
        "status": status,
        "exclusion_reasons": exclusion_reasons,
        "missing_details": missing_details
    }


def generate_eligibility_report(details):

    result = classify_pm_kisan(details)

    report = f"""
# SCHEMESENSE – ELIGIBILITY REPORT

## Applicant Information

**Applicant:** {details.get("Name", "Not provided")}

**Age:** {details.get("Age", "Not provided")}

**State:** {details.get("State", "Not provided")}

## Assessment Result

### {result["status"]}

## Possible Exclusion Reasons
"""

    if result["exclusion_reasons"]:
        for reason in result["exclusion_reasons"]:
            report += f"\n- {reason}"
    else:
        report += "\n- No listed exclusion detected from the answers."

    report += "\n\n## Missing Information\n"

    if result["missing_details"]:
        for item in result["missing_details"]:
            report += f"\n- {item}"
    else:
        report += "\n- No missing fields detected by this checklist."

    report += """

## Important Notice

This is an AI-assisted preliminary screening report.
It is not an official government eligibility certificate.

Verify all conditions, exceptions, land records and current
rules through the official PM-KISAN portal and authorities.
"""

    return report


print("Updated SchemeSense eligibility logic successfully!")


In [ ]:

# TEST 1: All answers provided, no listed exclusions

test_1 = {
    "Name": "Priya",
    "Age": 35,
    "State": "Andhra Pradesh",
    "Farmer family members": "4",
    "Agricultural land ownership": "Yes",
    "Land record available": "Yes",
    "Institutional landholding": "No",
    "Constitutional or elected position": "No",
    "Government employment": "No",
    "Monthly pension": 0,
    "Paid income tax last assessment year": "No",
    "Registered practicing professional": "No",
    "NRI status": "No"
}

# TEST 2: Income tax payer

test_2 = test_1.copy()
test_2["Paid income tax last assessment year"] = "Yes"

# TEST 3: Missing information

test_3 = test_1.copy()
test_3["NRI status"] = "Not sure"


for number, test in enumerate([test_1, test_2, test_3], start=1):
    print(f"\n========== TEST {number} ==========")
    result = classify_pm_kisan(test)
    print("Status:", result["status"])
    print("Exclusions:", result["exclusion_reasons"])
    print("Missing:", result["missing_details"])


In [ ]:

import os

pdf_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".pdf")
]

print("Available PDF files:")

for filename in pdf_files:
    print(filename)



In [ ]:

from pypdf import PdfReader

# Find the uploaded PMAY-G PDF
pmayg_files = [
    f for f in pdf_files
    if "pmay" in f.lower()
]

if not pmayg_files:
    print("PMAY-G PDF not found. Please upload it again.")

else:
    filename = pmayg_files[0]
    reader = PdfReader(filename)

    # Initialize documents if necessary
    if "documents" not in globals():
        documents = []

    # Remove previously extracted pages from this same PDF
    documents = [
        doc for doc in documents
        if doc.get("source") != filename
    ]

    added = 0

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text and page_text.strip():

            documents.append({
                "text": page_text,
                "source": filename,
                "page": page_number,
                "document_type": "official",
                "scheme": "PMAY-G"
            })

            added += 1

    print("PMAY-G PDF extracted successfully!")
    print("File:", filename)
    print("Pages extracted:", added)
    print("Total documents:", len(documents))


In [ ]:

from google.colab import files
from pypdf import PdfReader
import io

print("Select your PMAY-G Revised Phase-II Guidelines PDF")

uploaded_files = files.upload()

if not uploaded_files:
    print("No file selected. Please try again.")

else:
    filename, file_bytes = next(iter(uploaded_files.items()))

    if not filename.lower().endswith(".pdf"):
        print("Error: Please select a PDF file.")

    else:
        reader = PdfReader(io.BytesIO(file_bytes))

        if "documents" not in globals():
            documents = []

        # Remove duplicate pages from this same PDF
        documents = [
            doc for doc in documents
            if doc.get("source") != filename
        ]

        added = 0

        for page_number, page in enumerate(reader.pages, start=1):

            page_text = page.extract_text()

            if page_text and page_text.strip():

                documents.append({
                    "text": page_text,
                    "source": filename,
                    "page": page_number,
                    "document_type": "official",
                    "scheme": "PMAY-G"
                })

                added += 1

        print("\nPMAY-G PDF processing completed!")
        print("Filename:", filename)
        print("Pages extracted:", added)
        print("Total documents:", len(documents))


In [ ]:

# Identify the scheme for each document

for doc in documents:
    source_name = doc["source"].lower()

    if "pmay" in source_name:
        doc["scheme"] = "PMAY-G"
    elif "pm_kisan" in source_name or "pm-kisan" in source_name or "pmkisan" in source_name:
        doc["scheme"] = "PM-KISAN"

# Create text chunks
def create_chunks(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:
    scheme = doc.get("scheme")

    if scheme not in ["PM-KISAN", "PMAY-G"]:
        print("Skipped document with unknown scheme:", doc["source"])
        continue

    for chunk in create_chunks(doc["text"]):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "document_type": doc.get("document_type", "sample"),
            "scheme": scheme
        })

print("Updated text chunks created!")
print("Total chunks:", len(all_chunks))

print("\nScheme-wise chunk count:")

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(1 for chunk in all_chunks if chunk["scheme"] == scheme)
    print(scheme, ":", count)


In [ ]:

from openai import OpenAI
from google.colab import userdata

# Get API key securely from Colab Secrets
NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError(
        "NEXUS_API_KEY not found. Add it in Colab Secrets."
    )

# Connect to Nexus API
client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# Select working model
MODEL_NAME = "nova-micro"

print("Nexus API connected successfully!")
print("Selected model:", MODEL_NAME)

In [ ]:

from openai import OpenAI
from google.colab import userdata

# Read and clean API key
NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError("NEXUS_API_KEY is missing!")

NEXUS_API_KEY = NEXUS_API_KEY.strip()

# Connect to Nexus
client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

MODEL_NAME = "nova-micro"

print("Nexus API connection initialized successfully!")


In [ ]:

test_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=["SchemeSense government scheme assistant"]
)

print("Embedding API working!")
print("Embedding dimensions:",
      len(test_response.data[0].embedding))


In [ ]:

print("Documents available:", len(documents))
print("Text chunks available:", len(all_chunks))

print("\nScheme-wise document chunks:")

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(
        1 for chunk in all_chunks
        if chunk.get("scheme") == scheme
    )
    print(scheme, ":", count)


In [ ]:

import numpy as np
import faiss

texts = [chunk["text"] for chunk in all_chunks]

if not texts:
    raise ValueError("No text chunks found!")

# Generate embeddings in batches
all_embeddings = []
batch_size = 50

for start in range(0, len(texts), batch_size):

    batch = texts[start:start + batch_size]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    all_embeddings.extend([
        item.embedding for item in response.data
    ])

# Convert to NumPy
embeddings = np.array(
    all_embeddings,
    dtype="float32"
)

# Normalize
faiss.normalize_L2(embeddings)

# Build FAISS database
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS database rebuilt successfully!")
print("Total vectors:", index.ntotal)
print("Embedding dimensions:", embeddings.shape[1])


In [ ]:

def search_selected_scheme(question, selected_scheme, top_k=5):

    if selected_scheme not in ["PM-KISAN", "PMAY-G"]:
        raise ValueError("Please select a valid government scheme.")

    # Find chunks belonging to selected scheme
    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk.get("scheme") == selected_scheme
    ]

    if not selected_indices:
        return []

    # Create question embedding
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [query_response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    # Search selected scheme vectors only
    scheme_vectors = embeddings[selected_indices]

    temporary_index = faiss.IndexFlatIP(
        scheme_vectors.shape[1]
    )

    temporary_index.add(scheme_vectors)

    k = min(top_k, len(selected_indices))

    scores, indices = temporary_index.search(
        query_vector, k
    )

    results = []

    for position, score in zip(indices[0], scores[0]):

        if position < 0:
            continue

        original_index = selected_indices[position]

        result = all_chunks[original_index].copy()
        result["similarity_score"] = float(score)

        results.append(result)

    return results


print("Multi-scheme search function created successfully!")


In [ ]:

test_results = search_selected_scheme(
    "What are the eligibility conditions for PMAY-G?",
    "PMAY-G"
)

for result in test_results:

    print("\nSource:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["similarity_score"], 3))
    print("Text:", result["text"][:400])
    print("-" * 60)


In [ ]:

def generate_multi_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme,
        top_k=5
    )

    if not results:
        return "No relevant information found in the selected scheme documents."

    context = ""

    for result in results:
        context += f"""
Source: {result['source']}
Page: {result['page']}
Content: {result['text']}

"""

    prompt = f"""
You are SchemeSense, a government scheme information assistant.

Selected scheme: {selected_scheme}

Answer the user's question using ONLY the retrieved document context.

Instructions:
1. Use simple English.
2. Do not invent eligibility conditions.
3. If information is unavailable, say so clearly.
4. Mention the PDF filename and page number for important claims.
5. Do not declare final government eligibility.

Retrieved document context:
{context}

User question:
{question}

Provide a clear and structured answer.
"""

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": "You are a careful RAG-based government scheme assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=700
    )

    return response.choices[0].message.content


print("Multi-scheme AI answer function created successfully!")


In [ ]:

print("========== PMAY-G TEST ==========")

print(
    generate_multi_scheme_answer(
        "What is the objective of PMAY-G?",
        "PMAY-G"
    )
)

print("\n========== PM-KISAN TEST ==========")

print(
    generate_multi_scheme_answer(
        "How much financial assistance is provided under PM-KISAN?",
        "PM-KISAN"
    )
)


In [ ]:

print("========== DOCUMENT CHECK ==========")

print("Total documents:", len(documents))

for doc in documents:
    print(
        "File:", doc["source"],
        "| Scheme:", doc.get("scheme", "Not assigned")
    )

print("\n========== CHUNK CHECK ==========")

for scheme in ["PM-KISAN", "PMAY-G"]:
    matching_chunks = [
        chunk for chunk in all_chunks
        if chunk.get("scheme") == scheme
    ]

    print(scheme, "chunks:", len(matching_chunks))


In [ ]:

from google.colab import files
from pypdf import PdfReader
import io

print("Upload both PM-KISAN PDF documents")

uploaded_kisan = files.upload()

if not uploaded_kisan:
    print("No files uploaded. Please try again.")

else:
    added = 0

    existing_keys = {
        (doc["source"], doc["page"])
        for doc in documents
    }

    for filename, file_bytes in uploaded_kisan.items():

        if not filename.lower().endswith(".pdf"):
            print("Skipped non-PDF:", filename)
            continue

        reader = PdfReader(io.BytesIO(file_bytes))

        for page_number, page in enumerate(reader.pages, start=1):

            page_text = page.extract_text()

            key = (filename, page_number)

            if page_text and page_text.strip() and key not in existing_keys:

                documents.append({
                    "text": page_text,
                    "source": filename,
                    "page": page_number,
                    "document_type": "official",
                    "scheme": "PM-KISAN"
                })

                existing_keys.add(key)
                added += 1

    print("\nPM-KISAN restoration completed!")
    print("New pages added:", added)
    print("Total documents:", len(documents))


In [ ]:

all_chunks = []

for doc in documents:
    for chunk in create_chunks(
        doc["text"],
        chunk_size=150,
        overlap=30
    ):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "document_type": doc.get("document_type", "sample"),
            "scheme": doc.get("scheme", "PM-KISAN")
        })

print("Updated chunks created!")

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(
        1 for chunk in all_chunks
        if chunk["scheme"] == scheme
    )
    print(scheme, "chunks:", count)


In [ ]:

import numpy as np
import faiss

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = []
batch_size = 50

for start in range(0, len(texts), batch_size):

    batch = texts[start:start + batch_size]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    all_embeddings.extend([
        item.embedding for item in response.data
    ])

embeddings = np.array(all_embeddings, dtype="float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings generated successfully!")
print("Total vectors:", index.ntotal)
print("Vector dimensions:", embeddings.shape[1])


In [ ]:

def search_selected_scheme(question, selected_scheme, top_k=5):

    if selected_scheme not in ["PM-KISAN", "PMAY-G"]:
        raise ValueError("Select a valid scheme.")

    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk.get("scheme") == selected_scheme
    ]

    if not selected_indices:
        return []

    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [query_response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scheme_vectors = embeddings[selected_indices]

    temporary_index = faiss.IndexFlatIP(
        scheme_vectors.shape[1]
    )

    temporary_index.add(scheme_vectors)

    k = min(top_k, len(selected_indices))

    scores, positions = temporary_index.search(
        query_vector, k
    )

    results = []

    for position, score in zip(positions[0], scores[0]):

        if position < 0:
            continue

        original_index = selected_indices[position]

        result = all_chunks[original_index].copy()
        result["similarity_score"] = float(score)

        results.append(result)

    return results


print("Scheme-specific search function created!")


In [ ]:

results = search_selected_scheme(
    "How much financial assistance does PM-KISAN provide to farmers?",
    "PM-KISAN"
)

for result in results:
    print("\nScheme:", result["scheme"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["similarity_score"], 3))
    print("Text:", result["text"][:400])
    print("-" * 60)


In [ ]:

results_pmay = search_selected_scheme(
    "What is the main objective of PMAY-G?",
    "PMAY-G"
)

for result in results_pmay:
    print("\nScheme:", result["scheme"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["similarity_score"], 3))
    print("Text:", result["text"][:400])
    print("-" * 60)


In [ ]:

def generate_multi_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme,
        top_k=5
    )

    if not results:
        return (
            "I could not find relevant information in the "
            "uploaded documents for " + selected_scheme +
            ". Please upload the relevant scheme guidelines."
        )

    context = ""

    for i, result in enumerate(results, start=1):
        context += (
            f"\nPassage {i}\n"
            f"Scheme: {result['scheme']}\n"
            f"Source: {result['source']}\n"
            f"Page: {result['page']}\n"
            f"Text: {result['text']}\n"
        )

    prompt = f"""
You are SchemeSense, a government scheme information assistant.

Selected scheme: {selected_scheme}

Answer the user's question using ONLY the retrieved context below.

Rules:
1. Use simple English.
2. Do not invent facts, eligibility conditions or amounts.
3. If information is missing, clearly say it is not available
   in the uploaded documents.
4. Mention the source filename and page number.
5. Treat the answer as document-based information, not a
   final government eligibility decision.
6. Clearly distinguish sample documents from official guidelines.

Retrieved context:
{context}

User question:
{question}

Provide a clear and helpful answer.
"""

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": "You are a careful government scheme information assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


print("SchemeSense AI answer function created successfully!")


In [ ]:

question = "What is the main objective of PMAY-G?"

answer = generate_multi_scheme_answer(
    question,
    "PMAY-G"
)

print("SCHEMESENSE AI ANSWER")
print("=" * 50)
print(answer)


In [ ]:

question = "How much financial assistance does PM-KISAN provide to farmers?"

answer = generate_multi_scheme_answer(
    question,
    "PM-KISAN"
)

print("SCHEMESENSE PM-KISAN ANSWER")
print("=" * 50)
print(answer)


In [ ]:

import gradio as gr

def schemesense_multi_chat(message, history, selected_scheme):

    if not message.strip():
        return "Please enter your question."

    answer = generate_multi_scheme_answer(
        message,
        selected_scheme
    )

    return answer


with gr.Blocks(
    title="SchemeSense AI Assistant"
) as chatbot_app:

    gr.Markdown("""
    # 🏛️ SchemeSense
    ### AI-Based Government Scheme Information Assistant

    Ask questions about government schemes using uploaded
    reference documents.
    """)

    scheme_dropdown = gr.Dropdown(
        choices=["PM-KISAN", "PMAY-G"],
        value="PM-KISAN",
        label="Select Government Scheme"
    )

    gr.ChatInterface(
        fn=schemesense_multi_chat,
        additional_inputs=[scheme_dropdown],
        examples=[
            ["What is PM-KISAN?", "PM-KISAN"],
            ["How much financial assistance is provided?", "PM-KISAN"],
            ["What is the main objective?", "PMAY-G"],
            ["Who can benefit from this scheme?", "PMAY-G"]
        ]
    )


chatbot_app.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft()
)



In [ ]:

required_items = [
    "client",
    "MODEL_NAME",
    "documents",
    "all_chunks",
    "embeddings",
    "index",
    "search_selected_scheme",
    "generate_multi_scheme_answer",
    "classify_pm_kisan"
]

print("========== SCHEMESENSE SYSTEM CHECK ==========")

for item in required_items:
    exists = item in globals()

    print(
        item,
        ":",
        "AVAILABLE" if exists else "MISSING"
    )

print("\nTotal documents:", len(documents) if "documents" in globals() else 0)
print("Total chunks:", len(all_chunks) if "all_chunks" in globals() else 0)


In [ ]:

!pip install -q openai pypdf faiss-cpu gradio numpy

import numpy as np
import faiss
import gradio as gr
import io

from pypdf import PdfReader
from openai import OpenAI
from google.colab import files, userdata

# Read API key securely from Colab Secrets
NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError(
        "NEXUS_API_KEY is missing. Add it in Colab Secrets."
    )

NEXUS_API_KEY = NEXUS_API_KEY.strip()

# Connect to Nexus API
client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

MODEL_NAME = "nova-micro"

# Test connection
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Reply only: SchemeSense connected"}
    ]
)

print("API connection successful!")
print(response.choices[0].message.content)


In [ ]:

from google.colab import files
from pypdf import PdfReader
import io

print("Upload your SchemeSense government scheme PDFs")

uploaded_files = files.upload()

documents = []

for filename, file_bytes in uploaded_files.items():

    if not filename.lower().endswith(".pdf"):
        print("Skipped:", filename)
        continue

    reader = PdfReader(io.BytesIO(file_bytes))

    if "pmay" in filename.lower():
        scheme = "PMAY-G"
    elif "pm-kisan" in filename.lower().replace("_", "-") or "pmkisan" in filename.lower():
        scheme = "PM-KISAN"
    else:
        print("Scheme not recognized:", filename)
        continue

    added_pages = 0

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text and page_text.strip():

            documents.append({
                "text": page_text,
                "source": filename,
                "page": page_number,
                "document_type": (
                    "sample"
                    if "sample" in filename.lower()
                    else "official"
                ),
                "scheme": scheme
            })

            added_pages += 1

    print(
        filename,
        "| Scheme:", scheme,
        "| Pages extracted:", added_pages
    )

print("\nTotal documents extracted:", len(documents))


In [ ]:

from google.colab import files
from pypdf import PdfReader
import io

print("Upload your PMAY-G guidelines PDF")

uploaded_pmay = files.upload()

for filename, file_bytes in uploaded_pmay.items():

    if not filename.lower().endswith(".pdf"):
        print("Skipped non-PDF:", filename)
        continue

    reader = PdfReader(io.BytesIO(file_bytes))

    added_pages = 0

    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text()

        if page_text and page_text.strip():

            documents.append({
                "text": page_text,
                "source": filename,
                "page": page_number,
                "document_type": "official",
                "scheme": "PMAY-G"
            })

            added_pages += 1

    print("\nFile:", filename)
    print("Scheme: PMAY-G")
    print("Pages extracted:", added_pages)

print("\nTotal documents:", len(documents))


In [ ]:

def create_chunks(text, chunk_size=150, overlap=30):

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


all_chunks = []

for doc in documents:

    text_chunks = create_chunks(
        doc["text"],
        chunk_size=150,
        overlap=30
    )

    for chunk in text_chunks:

        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "scheme": doc["scheme"],
            "document_type": doc["document_type"]
        })


print("========== CHUNK CREATION ==========")
print("Total documents:", len(documents))
print("Total chunks:", len(all_chunks))

for scheme in ["PM-KISAN", "PMAY-G"]:

    count = sum(
        1 for chunk in all_chunks
        if chunk["scheme"] == scheme
    )

    print(scheme, "chunks:", count)


In [ ]:

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = []

for start in range(0, len(texts), 50):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts[start:start + 50]
    )

    all_embeddings.extend(
        [item.embedding for item in response.data]
    )

embeddings = np.array(all_embeddings, dtype="float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings created successfully!")
print("Total vectors:", index.ntotal)


In [ ]:

def create_chunks(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

    return chunks


all_chunks = []

for doc in documents:
    for chunk in create_chunks(doc["text"]):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "scheme": doc["scheme"],
            "document_type": doc["document_type"]
        })

print("Total documents:", len(documents))
print("Total chunks:", len(all_chunks))

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(1 for c in all_chunks if c["scheme"] == scheme)
    print(scheme, "chunks:", count)


In [ ]:

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = []

for start in range(0, len(texts), 50):
    batch = texts[start:start + 50]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    all_embeddings.extend(
        [item.embedding for item in response.data]
    )

embeddings = np.array(all_embeddings, dtype="float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings generated successfully!")
print("Total vectors:", index.ntotal)
print("Dimensions:", embeddings.shape[1])


In [ ]:

def search_selected_scheme(question, selected_scheme, top_k=5):

    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk["scheme"] == selected_scheme
    ]

    if not selected_indices:
        return []

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scheme_vectors = embeddings[selected_indices]

    temp_index = faiss.IndexFlatIP(
        scheme_vectors.shape[1]
    )

    temp_index.add(scheme_vectors)

    k = min(top_k, len(selected_indices))

    scores, positions = temp_index.search(query_vector, k)

    results = []

    for position, score in zip(positions[0], scores[0]):

        if position >= 0:
            original_index = selected_indices[position]

            result = all_chunks[original_index].copy()
            result["similarity_score"] = float(score)

            results.append(result)

    return results


print("SchemeSense search function created successfully!")


In [ ]:

def generate_multi_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme,
        top_k=5
    )

    if not results:
        return "No relevant information found in the uploaded documents."

    context = ""

    for i, result in enumerate(results, start=1):
        context += f"""
Passage {i}
Scheme: {result['scheme']}
Source: {result['source']}
Page: {result['page']}
Document type: {result['document_type']}
Text: {result['text']}
"""

    prompt = f"""
You are SchemeSense, a government scheme information assistant.

Selected scheme: {selected_scheme}

Answer using ONLY the retrieved document context.

Rules:
1. Use simple English.
2. Do not invent facts or eligibility rules.
3. If information is missing, say so clearly.
4. Mention the source filename and page number.
5. Do not claim final government eligibility or approval.
6. Clearly identify sample documents as samples, not official guidelines.

Retrieved context:
{context}

User question:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a careful government scheme information assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


print("SchemeSense AI answer function created successfully!")


In [ ]:

answer = generate_multi_scheme_answer(
    "What is the main objective of this scheme?",
    "PMAY-G"
)

print(answer)


In [ ]:

def classify_pm_kisan(details):

    exclusion_reasons = []
    missing_details = []

    required_fields = [
        "Name",
        "State",
        "Age",
        "Farmer family members",
        "Agricultural land ownership",
        "Land record available",
        "Institutional landholding",
        "Constitutional or elected position",
        "Government employment",
        "Paid income tax last assessment year",
        "Registered practicing professional",
        "NRI status",
        "Monthly pension"
    ]

    for field in required_fields:
        if details.get(field) in [None, ""]:
            missing_details.append(f"{field} is missing.")

    yes_no_fields = required_fields[4:12]

    for field in yes_no_fields:
        answer = str(details.get(field, "")).strip().lower()
        if answer not in ["yes", "no"]:
            if f"{field} is missing." not in missing_details:
                missing_details.append(f"Please enter Yes or No for {field}.")

    if str(details.get("Agricultural land ownership", "")).lower() == "no":
        exclusion_reasons.append("Agricultural land ownership requirement may not be met.")

    exclusion_fields = [
        "Institutional landholding",
        "Constitutional or elected position",
        "Government employment",
        "Paid income tax last assessment year",
        "Registered practicing professional",
        "NRI status"
    ]

    for field in exclusion_fields:
        if str(details.get(field, "")).strip().lower() == "yes":
            exclusion_reasons.append(f"Possible exclusion: {field}.")

    try:
        pension = float(details.get("Monthly pension"))
        if pension < 0:
            missing_details.append("Monthly pension cannot be negative.")
        elif pension >= 10000:
            exclusion_reasons.append("Monthly pension is ₹10,000 or more; official rules and exceptions must be checked.")
    except (ValueError, TypeError):
        if "Monthly pension is missing." not in missing_details:
            missing_details.append("Enter a valid monthly pension amount.")

    if missing_details:
        status = "MORE INFORMATION REQUIRED"
    elif exclusion_reasons:
        status = "POSSIBLE EXCLUSION – OFFICIAL REVIEW REQUIRED"
    else:
        status = "NO LISTED EXCLUSION DETECTED – OFFICIAL VERIFICATION REQUIRED"

    return {
        "status": status,
        "exclusion_reasons": exclusion_reasons,
        "missing_details": missing_details
    }


print("PM-KISAN eligibility checker created successfully!")


In [ ]:

sample_details = {
    "Name": "Test Applicant",
    "State": "Tamil Nadu",
    "Age": 35,
    "Farmer family members": "2",
    "Agricultural land ownership": "yes",
    "Land record available": "yes",
    "Institutional landholding": "no",
    "Constitutional or elected position": "no",
    "Government employment": "no",
    "Paid income tax last assessment year": "no",
    "Registered practicing professional": "no",
    "NRI status": "no",
    "Monthly pension": 0
}

result = classify_pm_kisan(sample_details)

print("Eligibility Screening Result")
print("----------------------------")
print("Status:", result["status"])
print("Possible exclusion reasons:", result["exclusion_reasons"])
print("Missing details:", result["missing_details"])


In [ ]:

document_info = []

for doc in documents:
    document_info.append({
        "File Name": doc["source"],
        "Scheme": doc["scheme"],
        "Document Type": doc["document_type"],
        "Pages": 1
    })

import pandas as pd

documents_df = pd.DataFrame(document_info)

print("SchemeSense Document Records")
display(documents_df)

print("\nTotal documents:", len(documents_df))
print("Total chunks:", len(all_chunks))


In [ ]:

import pandas as pd

grouped_docs = {}

for doc in documents:

    source = doc.get("source")
    scheme = doc.get("scheme")
    doc_type = doc.get("document_type")

    if not source or not scheme:
        continue

    key = (source, scheme, doc_type)

    if key not in grouped_docs:
        grouped_docs[key] = set()

    grouped_docs[key].add(doc.get("page", 1))

document_info = []

for (source, scheme, doc_type), pages in grouped_docs.items():
    document_info.append({
        "File Name": source,
        "Scheme": scheme,
        "Document Type": doc_type,
        "Pages": len(pages)
    })

documents_df = pd.DataFrame(document_info)

print("SchemeSense Document Records")
display(documents_df)

print("Total unique documents:", len(documents_df))


In [ ]:

import gradio as gr

with gr.Blocks(title="SchemeSense AI Dashboard") as dashboard:

    gr.Markdown("""
    # 🏛️ SCHEMESENSE
    ### AI-Based Government Scheme Information & Eligibility Assistant

    Discover government schemes, ask questions and check preliminary eligibility.

    ---
    """)

    with gr.Tab("🏠 Home"):
        gr.Markdown("""
        ## Welcome to SchemeSense!

        Your AI-powered assistant for government scheme information.

        **Available Schemes**
        - PM-KISAN – Farmer income support information
        - PMAY-G – Rural housing scheme information

        **Dashboard Features**
        - AI-powered document question answering
        - Preliminary PM-KISAN eligibility screening
        - Government scheme document information
        """)

    with gr.Tab("🤖 AI Chatbot"):
        gr.Markdown("## Ask Questions About Government Schemes")

        scheme_dropdown = gr.Dropdown(
            choices=["PM-KISAN", "PMAY-G"],
            value="PM-KISAN",
            label="Select Scheme"
        )

        gr.ChatInterface(
            fn=lambda message, history, scheme:
                generate_multi_scheme_answer(message, scheme),
            additional_inputs=[scheme_dropdown]
        )

    with gr.Tab("📋 Eligibility Checker"):
        gr.Markdown("## PM-KISAN Preliminary Eligibility Screening")

        gr.Markdown("Enter fictional test details. Do not enter sensitive personal information.")

        name = gr.Textbox(label="Applicant Name")
        state = gr.Textbox(label="State")
        age = gr.Number(label="Age", precision=0)
        family = gr.Textbox(label="Farmer Family Members")

        land = gr.Radio(["yes", "no"], label="Agricultural Land Ownership")
        records = gr.Radio(["yes", "no"], label="Land Record Available")
        institutional = gr.Radio(["yes", "no"], label="Institutional Landholding")
        position = gr.Radio(["yes", "no"], label="Constitutional or Elected Position")
        government = gr.Radio(["yes", "no"], label="Government Employment")
        tax = gr.Radio(["yes", "no"], label="Paid Income Tax Last Assessment Year")
        professional = gr.Radio(["yes", "no"], label="Registered Practicing Professional")
        nri = gr.Radio(["yes", "no"], label="NRI Status")
        pension = gr.Number(label="Monthly Pension (₹)", value=0)

        check_button = gr.Button("Check Preliminary Eligibility")
        eligibility_output = gr.Textbox(
            label="Screening Result",
            lines=8
        )

        def run_eligibility(name, state, age, family, land, records,
                            institutional, position, government, tax,
                            professional, nri, pension):

            details = {
                "Name": name,
                "State": state,
                "Age": age,
                "Farmer family members": family,
                "Agricultural land ownership": land,
                "Land record available": records,
                "Institutional landholding": institutional,
                "Constitutional or elected position": position,
                "Government employment": government,
                "Paid income tax last assessment year": tax,
                "Registered practicing professional": professional,
                "NRI status": nri,
                "Monthly pension": pension
            }

            result = classify_pm_kisan(details)

            return (
                f"Status: {result['status']}\n\n"
                f"Possible exclusions: {result['exclusion_reasons']}\n\n"
                f"Missing details: {result['missing_details']}\n\n"
                "This is preliminary screening only. "
                "Final eligibility must be verified by the government."
            )

        check_button.click(
            fn=run_eligibility,
            inputs=[
                name, state, age, family, land, records,
                institutional, position, government, tax,
                professional, nri, pension
            ],
            outputs=eligibility_output
        )

    with gr.Tab("📂 Documents"):
        gr.Markdown("## Uploaded Scheme Documents")
        gr.Dataframe(
            value=documents_df,
            interactive=False,
            label="Document Information"
        )

    with gr.Tab("ℹ️ About"):
        gr.Markdown("""
        ## About SchemeSense

        SchemeSense is an AI-based government scheme information
        and preliminary eligibility assistant.

        **Technologies Used**
        - Python
        - Google Colab
        - FAISS Vector Database
        - Text Embeddings
        - Nexus AI API
        - Gradio

        **Important Notice**

        This is an academic prototype. Its responses depend on
        uploaded documents. It does not provide official eligibility
        decisions or guarantee government benefits.
        """)

print("SchemeSense dashboard created successfully!")


In [ ]:

dashboard.launch(
    share=True,
    debug=False,
    theme=gr.themes.Soft()
)


In [ ]:

import gradio as gr

professional_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
    neutral_hue="slate"
)

print("Professional SchemeSense theme created!")


Professional SchemeSense theme created!


In [ ]:

with gr.Blocks(
    title="SchemeSense | Smart Government Portal",
    theme=professional_theme,
    css="""
    .gradio-container {
        max-width: 1250px !important;
        margin: auto !important;
    }
    .main-title {
        text-align: center;
        color: #174A7E;
        padding: 15px;
    }
    .stat-card {
        text-align: center;
        border-radius: 12px;
        padding: 15px;
    }
    """
) as professional_dashboard:

    gr.Markdown("""
    <div class="main-title">
        <h1>🏛️ SCHEMESENSE</h1>
        <h3>Smart Government Scheme Information Portal</h3>
        <p>AI-powered scheme assistance and preliminary eligibility screening</p>
    </div>
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Dashboard Overview")
            gr.Markdown("Welcome to SchemeSense. Access government scheme information through AI.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📚 02")
            gr.Markdown("**Available Schemes**")
        with gr.Column():
            gr.Markdown(f"### 📄 {len(documents_df)}")
            gr.Markdown("**Unique Documents**")
        with gr.Column():
            gr.Markdown(f"### 🔍 {len(all_chunks)}")
            gr.Markdown("**Knowledge Chunks**")

    gr.Markdown("---")
    gr.Markdown("## Available Government Schemes")

    with gr.Row():
        with gr.Column():
            gr.Markdown("""
            ### 🌾 PM-KISAN
            Farmer income support scheme information.

            **Feature:** Preliminary eligibility screening.
            """)
        with gr.Column():
            gr.Markdown("""
            ### 🏠 PMAY-G
            Rural housing scheme information.

            **Feature:** AI document-based question answering.
            """)

    gr.Markdown("---")
    gr.Markdown("""
    **Technology Stack:** Python | FAISS | Embeddings | Nexus AI | Gradio

    *Academic prototype – not an official government portal.*
    """)

print("Professional home page created successfully!")


/tmp/ipykernel_49970/1532438300.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Professional home page created successfully!


In [ ]:

required_items = [
    "generate_multi_scheme_answer",
    "classify_pm_kisan",
    "documents_df",
    "all_chunks"
]

for item in required_items:
    print(item, ":", "Available" if item in globals() else "Missing")


generate_multi_scheme_answer : Available
classify_pm_kisan : Available
documents_df : Available
all_chunks : Available


In [ ]:

with professional_dashboard:

    with gr.Tab("🤖 AI Chatbot"):

        gr.Markdown("""
        ## SchemeSense AI Assistant
        Ask questions about government schemes using uploaded documents.
        """)

        chatbot_scheme = gr.Dropdown(
            choices=["PM-KISAN", "PMAY-G"],
            value="PM-KISAN",
            label="Choose Government Scheme"
        )

        chatbot = gr.ChatInterface(
            fn=lambda message, history, scheme:
                generate_multi_scheme_answer(message, scheme),
            additional_inputs=[chatbot_scheme],
            examples=[
                ["What is the objective of this scheme?", "PMAY-G"],
                ["Who can benefit from this scheme?", "PMAY-G"],
                ["What is PM-KISAN?", "PM-KISAN"]
            ]
        )

print("AI Chatbot tab added!")


AI Chatbot tab added!


In [ ]:

with professional_dashboard:

    # ELIGIBILITY CHECKER
    with gr.Tab("📋 Eligibility Checker"):

        gr.Markdown("""
        ## PM-KISAN Eligibility Screening
        Enter applicant details for preliminary screening.
        """)

        with gr.Row():
            name = gr.Textbox(label="Applicant Name")
            state = gr.Textbox(label="State")

        with gr.Row():
            age = gr.Number(label="Age", precision=0)
            family = gr.Textbox(label="Farmer Family Members")

        land = gr.Radio(["yes", "no"], label="Agricultural Land Ownership")
        records = gr.Radio(["yes", "no"], label="Land Record Available")
        institutional = gr.Radio(["yes", "no"], label="Institutional Landholding")
        position = gr.Radio(["yes", "no"], label="Constitutional or Elected Position")
        government = gr.Radio(["yes", "no"], label="Government Employment")
        tax = gr.Radio(["yes", "no"], label="Paid Income Tax Last Assessment Year")
        professional = gr.Radio(["yes", "no"], label="Registered Practicing Professional")
        nri = gr.Radio(["yes", "no"], label="NRI Status")
        pension = gr.Number(label="Monthly Pension (₹)", value=0)

        check_btn = gr.Button(
            "Check Eligibility",
            variant="primary"
        )

        eligibility_result = gr.Textbox(
            label="Screening Result",
            lines=8
        )

        def professional_eligibility(
            name, state, age, family, land, records,
            institutional, position, government, tax,
            professional, nri, pension
        ):
            details = {
                "Name": name,
                "State": state,
                "Age": age,
                "Farmer family members": family,
                "Agricultural land ownership": land,
                "Land record available": records,
                "Institutional landholding": institutional,
                "Constitutional or elected position": position,
                "Government employment": government,
                "Paid income tax last assessment year": tax,
                "Registered practicing professional": professional,
                "NRI status": nri,
                "Monthly pension": pension
            }

            result = classify_pm_kisan(details)

            return (
                f"STATUS: {result['status']}\n\n"
                f"Possible exclusions: {result['exclusion_reasons']}\n\n"
                f"Missing details: {result['missing_details']}\n\n"
                "This is preliminary screening only. "
                "Official verification is required."
            )

        check_btn.click(
            professional_eligibility,
            inputs=[
                name, state, age, family, land, records,
                institutional, position, government, tax,
                professional, nri, pension
            ],
            outputs=eligibility_result
        )

    # DOCUMENTS
    with gr.Tab("📂 Documents"):

        gr.Markdown("## Uploaded Government Scheme Documents")

        gr.Dataframe(
            value=documents_df,
            interactive=False,
            label="Document Information"
        )

        gr.Markdown("""
        **Note:** The PM-KISAN document is a sample reference.
        Verify scheme conditions using current official government guidelines.
        """)

    # ABOUT
    with gr.Tab("ℹ️ About"):

        gr.Markdown("""
        # About SchemeSense

        SchemeSense is an AI-based government scheme information
        and preliminary eligibility assistant.

        ### Technologies Used
        - Python
        - Google Colab
        - FAISS Vector Database
        - Text Embeddings
        - Nexus AI API
        - Gradio

        ### Main Features
        - Retrieval-Augmented Generation (RAG)
        - AI-powered scheme question answering
        - Preliminary eligibility screening
        - Document-based information retrieval

        ### Disclaimer
        This is an academic prototype, not an official government portal.
        It does not guarantee eligibility or benefit approval.
        """)

print("Eligibility Checker, Documents and About tabs added!")


Eligibility Checker, Documents and About tabs added!


In [ ]:


required_items = [
    "professional_dashboard",
    "professional_theme",
    "generate_multi_scheme_answer",
    "classify_pm_kisan",
    "documents_df",
    "all_chunks"
]

for item in required_items:
    print(item, ":", "Available" if item in globals() else "Missing")


professional_dashboard : Missing
professional_theme : Missing
generate_multi_scheme_answer : Missing
classify_pm_kisan : Missing
documents_df : Missing
all_chunks : Missing


In [ ]:


!pip install -q openai pypdf faiss-cpu gradio numpy pandas

import numpy as np
import pandas as pd
import faiss
import gradio as gr
import io

from pypdf import PdfReader
from openai import OpenAI
from google.colab import files, userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError("NEXUS_API_KEY is missing from Colab Secrets.")

client = OpenAI(
    api_key=NEXUS_API_KEY.strip(),
    base_url="https://nexusapi.navigatelabs.ai"
)

MODEL_NAME = "nova-micro"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Reply only: SchemeSense connected"}
    ]
)

print("API connection successful!")
print(response.choices[0].message.content)


API connection successful!
SchemeSense connected. How can I assist you further today?


In [ ]:

print("Upload your two SchemeSense PDF documents:")
print("1. PM-KISAN Sample Reference PDF")
print("2. PMAY-G Guidelines PDF")

uploaded_files = files.upload()

documents = []

for filename, file_data in uploaded_files.items():

    reader = PdfReader(io.BytesIO(file_data))

    scheme = (
        "PM-KISAN" if "KISAN" in filename.upper()
        else "PMAY-G" if "PMAY" in filename.upper()
        else "UNKNOWN"
    )

    document_type = (
        "sample" if scheme == "PM-KISAN"
        else "official" if scheme == "PMAY-G"
        else "unclassified"
    )

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        documents.append({
            "source": filename,
            "scheme": scheme,
            "document_type": document_type,
            "page": page_number,
            "text": text
        })

print("\nPDF restoration successful!")
print("Total pages:", len(documents))

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(1 for d in documents if d["scheme"] == scheme)
    print(scheme, "pages:", count)


Upload your two SchemeSense PDF documents:
1. PM-KISAN Sample Reference PDF
2. PMAY-G Guidelines PDF


Saving SchemeSense_PM_KISAN_Sample_Reference.pdf to SchemeSense_PM_KISAN_Sample_Reference.pdf

PDF restoration successful!
Total pages: 1
PM-KISAN pages: 1
PMAY-G pages: 0


In [ ]:


uploaded_pmay = files.upload()

for filename, file_data in uploaded_pmay.items():

    reader = PdfReader(io.BytesIO(file_data))

    for page_number, page in enumerate(reader.pages, start=1):

        documents.append({
            "source": filename,
            "scheme": "PMAY-G",
            "document_type": "official",
            "page": page_number,
            "text": page.extract_text() or ""
        })

print("PMAY-G PDF restored successfully!")
print("Total pages:", len(documents))

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(1 for d in documents if d["scheme"] == scheme)
    print(scheme, "pages:", count)


Saving 20250311250605284.pdf to 20250311250605284.pdf
PMAY-G PDF restored successfully!
Total pages: 23
PM-KISAN pages: 1
PMAY-G pages: 22


In [ ]:

def create_chunks(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

    return chunks


all_chunks = []

for doc in documents:
    for chunk in create_chunks(doc["text"]):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "scheme": doc["scheme"],
            "document_type": doc["document_type"]
        })

print("Total documents/pages:", len(documents))
print("Total chunks:", len(all_chunks))

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(1 for c in all_chunks if c["scheme"] == scheme)
    print(scheme, "chunks:", count)


Total documents/pages: 23
Total chunks: 43
PM-KISAN chunks: 3
PMAY-G chunks: 40


In [ ]:

texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = []

for start in range(0, len(texts), 50):

    batch = texts[start:start + 50]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    all_embeddings.extend(
        [item.embedding for item in response.data]
    )

embeddings = np.array(all_embeddings, dtype="float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings generated successfully!")
print("Total vectors:", index.ntotal)
print("Dimensions:", embeddings.shape[1])


Embeddings generated successfully!
Total vectors: 43
Dimensions: 1536


In [ ]:

def search_selected_scheme(question, selected_scheme, top_k=5):

    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk["scheme"] == selected_scheme
    ]

    if not selected_indices:
        return []

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scheme_vectors = embeddings[selected_indices]

    temp_index = faiss.IndexFlatIP(
        scheme_vectors.shape[1]
    )

    temp_index.add(scheme_vectors)

    k = min(top_k, len(selected_indices))

    scores, positions = temp_index.search(query_vector, k)

    results = []

    for position, score in zip(positions[0], scores[0]):

        if position >= 0:
            original_index = selected_indices[position]

            result = all_chunks[original_index].copy()
            result["similarity_score"] = float(score)

            results.append(result)

    return results


print("Document search function restored successfully!")


Document search function restored successfully!


In [ ]:

def generate_multi_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme,
        top_k=5
    )

    if not results:
        return "No relevant information found in the uploaded documents."

    context = ""

    for i, result in enumerate(results, start=1):
        context += f"""
Passage {i}
Scheme: {result['scheme']}
Source: {result['source']}
Page: {result['page']}
Document type: {result['document_type']}
Text: {result['text']}
"""

    prompt = f"""
You are SchemeSense, a government scheme information assistant.

Selected scheme: {selected_scheme}

Answer using ONLY the retrieved document context.

Rules:
1. Use simple English.
2. Do not invent facts or eligibility rules.
3. If information is missing, say so clearly.
4. Mention the source filename and page number.
5. Do not claim final government eligibility or approval.
6. Clearly identify sample documents as samples, not official guidelines.

Retrieved context:
{context}

User question:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a careful government scheme information assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


print("AI Chatbot function restored successfully!")


AI Chatbot function restored successfully!


In [ ]:

def classify_pm_kisan(details):

    exclusion_reasons = []
    missing_details = []

    required_fields = [
        "Name", "State", "Age", "Farmer family members",
        "Agricultural land ownership", "Land record available",
        "Institutional landholding", "Constitutional or elected position",
        "Government employment", "Paid income tax last assessment year",
        "Registered practicing professional", "NRI status",
        "Monthly pension"
    ]

    for field in required_fields:
        if details.get(field) in [None, ""]:
            missing_details.append(f"{field} is missing.")

    yes_no_fields = required_fields[4:12]

    for field in yes_no_fields:
        answer = str(details.get(field, "")).strip().lower()
        if answer not in ["yes", "no"]:
            if f"{field} is missing." not in missing_details:
                missing_details.append(f"Please confirm: {field}.")

    if str(details.get("Agricultural land ownership", "")).lower() == "no":
        exclusion_reasons.append("Agricultural land ownership requirement may not be met.")

    exclusion_fields = [
        "Institutional landholding",
        "Constitutional or elected position",
        "Government employment",
        "Paid income tax last assessment year",
        "Registered practicing professional",
        "NRI status"
    ]

    for field in exclusion_fields:
        if str(details.get(field, "")).strip().lower() == "yes":
            exclusion_reasons.append(f"Possible exclusion: {field}.")

    try:
        pension = float(details.get("Monthly pension"))
        if pension < 0:
            missing_details.append("Monthly pension cannot be negative.")
        elif pension >= 10000:
            exclusion_reasons.append(
                "Monthly pension is ₹10,000 or more; official rules and exceptions must be checked."
            )
    except (ValueError, TypeError):
        if "Monthly pension is missing." not in missing_details:
            missing_details.append("Enter a valid monthly pension amount.")

    if missing_details:
        status = "MORE INFORMATION REQUIRED"
    elif exclusion_reasons:
        status = "POSSIBLE EXCLUSION – OFFICIAL REVIEW REQUIRED"
    else:
        status = "NO LISTED EXCLUSION DETECTED – OFFICIAL VERIFICATION REQUIRED"

    return {
        "status": status,
        "exclusion_reasons": exclusion_reasons,
        "missing_details": missing_details
    }


print("Eligibility Checker restored successfully!")


Eligibility Checker restored successfully!


In [ ]:

# Create unique document records

grouped_docs = {}

for doc in documents:

    key = (
        doc["source"],
        doc["scheme"],
        doc["document_type"]
    )

    if key not in grouped_docs:
        grouped_docs[key] = set()

    grouped_docs[key].add(doc["page"])


document_info = []

for (source, scheme, doc_type), pages in grouped_docs.items():

    document_info.append({
        "File Name": source,
        "Scheme": scheme,
        "Document Type": doc_type,
        "Pages": len(pages)
    })


documents_df = pd.DataFrame(document_info)

# Create professional theme

professional_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
    neutral_hue="slate"
)

print("Documents table restored!")
display(documents_df)

print("\nProfessional theme restored!")
print("Total unique documents:", len(documents_df))
print("Total knowledge chunks:", len(all_chunks))


Documents table restored!


,File Name,Scheme,Document Type,Pages
0,SchemeSense_PM_KISAN_Sample_Reference.pdf,PM-KISAN,sample,1
1,20250311250605284.pdf,PMAY-G,official,22



Professional theme restored!
Total unique documents: 2
Total knowledge chunks: 43


In [ ]:

# ==========================================
# SCHEMESENSE - PROFESSIONAL AI DASHBOARD
# ==========================================

def chatbot_response(message, history, scheme):
    return generate_multi_scheme_answer(message, scheme)


def eligibility_result(
    name, state, age, family_members, land_ownership,
    land_record, institutional_land, constitutional_position,
    government_job, income_tax, professional, nri, pension
):
    details = {
        "Name": name,
        "State": state,
        "Age": age,
        "Farmer family members": family_members,
        "Agricultural land ownership": land_ownership,
        "Land record available": land_record,
        "Institutional landholding": institutional_land,
        "Constitutional or elected position": constitutional_position,
        "Government employment": government_job,
        "Paid income tax last assessment year": income_tax,
        "Registered practicing professional": professional,
        "NRI status": nri,
        "Monthly pension": pension
    }

    result = classify_pm_kisan(details)

    output = f"""
## Eligibility Screening Result

**Applicant:** {name or "Not provided"}

**State:** {state or "Not provided"}

### Status: {result['status']}

"""

    if result["missing_details"]:
        output += "\n### Information required\n"
        for item in result["missing_details"]:
            output += f"- {item}\n"

    if result["exclusion_reasons"]:
        output += "\n### Possible exclusion reasons\n"
        for item in result["exclusion_reasons"]:
            output += f"- {item}\n"

    output += """
---
**Disclaimer:** This is a preliminary AI-based screening result,
not official government eligibility confirmation or approval.
Please verify all details with the official PM-KISAN authorities.
"""
    return output


# Professional dashboard

with gr.Blocks(
    title="SchemeSense | Government Scheme Assistant"
) as professional_dashboard:

    gr.Markdown("""
    # 🏛️ SCHEMESENSE
    ### AI-Based Government Scheme Information & Eligibility Assistant

    Discover government schemes, ask questions and check
    preliminary eligibility using uploaded scheme documents.

    ---
    """)

    # HOME TAB
    with gr.Tab("🏠 Home"):

        gr.Markdown("""
        ## Welcome to SchemeSense

        Your AI-powered assistant for understanding government schemes.

        ### Dashboard Overview
        """)

        with gr.Row():
            with gr.Column():
                gr.Markdown("""
                ### 📄 2

                Uploaded Documents
                """)
            with gr.Column():
                gr.Markdown("""
                ### 📚 43

                Knowledge Chunks
                """)
            with gr.Column():
                gr.Markdown("""
                ### 🤖 AI + RAG

                Technology
                """)

        gr.Markdown("""
        ---

        ### Available Services

        - 💬 **AI Chatbot:** Ask questions about government schemes.
        - 📝 **Eligibility Checker:** Perform preliminary PM-KISAN screening.
        - 📂 **Documents:** View uploaded scheme reference documents.
        - ℹ️ **About:** Learn about the SchemeSense project.

        Select a tab above to get started.
        """)

    # AI CHATBOT TAB
    with gr.Tab("💬 AI Chatbot"):

        gr.Markdown("""
        ## SchemeSense AI Chatbot

        Ask questions about PM-KISAN or PMAY-G.
        Answers are generated using retrieved document information.
        """)

        chatbot_scheme = gr.Dropdown(
            choices=["PM-KISAN", "PMAY-G"],
            value="PM-KISAN",
            label="Select Government Scheme"
        )

        gr.ChatInterface(
            fn=chatbot_response,
            additional_inputs=[chatbot_scheme],
            examples=[
                ["What is PM-KISAN?", "PM-KISAN"],
                ["Explain the scheme benefits.", "PM-KISAN"],
                ["What is PMAY-G?", "PMAY-G"],
                ["What documents are mentioned in the guidelines?", "PMAY-G"]
            ]
        )

    # ELIGIBILITY CHECKER TAB
    with gr.Tab("📝 Eligibility Checker"):

        gr.Markdown("""
        ## PM-KISAN Preliminary Eligibility Checker

        Enter applicant information to receive a preliminary screening.
        Use fictional test information while demonstrating this project.
        """)

        with gr.Row():
            name = gr.Textbox(label="Applicant Name")
            state = gr.Textbox(label="State")

        with gr.Row():
            age = gr.Textbox(label="Age")
            family_members = gr.Textbox(
                label="Farmer family members"
            )

        land_ownership = gr.Dropdown(
            ["Yes", "No"],
            label="Agricultural land ownership"
        )

        land_record = gr.Dropdown(
            ["Yes", "No"],
            label="Land record available"
        )

        institutional_land = gr.Dropdown(
            ["Yes", "No"],
            label="Institutional landholding"
        )

        constitutional_position = gr.Dropdown(
            ["Yes", "No"],
            label="Constitutional or elected position"
        )

        government_job = gr.Dropdown(
            ["Yes", "No"],
            label="Government employment"
        )

        income_tax = gr.Dropdown(
            ["Yes", "No"],
            label="Paid income tax last assessment year"
        )

        professional = gr.Dropdown(
            ["Yes", "No"],
            label="Registered practicing professional"
        )

        nri = gr.Dropdown(
            ["Yes", "No"],
            label="NRI status"
        )

        pension = gr.Textbox(
            label="Monthly pension amount (₹)"
        )

        check_button = gr.Button(
            "Check Preliminary Eligibility",
            variant="primary"
        )

        eligibility_output = gr.Markdown()

        check_button.click(
            fn=eligibility_result,
            inputs=[
                name, state, age, family_members,
                land_ownership, land_record,
                institutional_land, constitutional_position,
                government_job, income_tax,
                professional, nri, pension
            ],
            outputs=eligibility_output
        )

    # DOCUMENTS TAB
    with gr.Tab("📂 Documents"):

        gr.Markdown("""
        ## Uploaded Government Scheme Documents

        These documents form the knowledge base used by SchemeSense.
        """)

        gr.Dataframe(
            value=documents_df,
            interactive=False,
            label="Document Information"
        )

        gr.Markdown("""
        ### Document Processing

        - PDF text extraction
        - Text chunking
        - Embedding generation
        - FAISS vector indexing
        - Retrieval-Augmented Generation (RAG)
        """)

    # ABOUT TAB
    with gr.Tab("ℹ️ About"):

        gr.Markdown("""
        ## About SchemeSense

        SchemeSense is an AI-based government scheme information
        and preliminary eligibility assistant.

        ### Project Objectives

        - Make government scheme information easier to understand.
        - Answer questions using uploaded scheme documents.
        - Support preliminary eligibility screening.
        - Provide document-based responses with source information.

        ### Technologies Used

        | Technology | Purpose |
        |---|---|
        | Python | Backend development |
        | Gradio | User interface |
        | FAISS | Vector database |
        | Embeddings | Text representation |
        | RAG | Document-based question answering |
        | Nexus API | AI model and embeddings |

        ### Important Disclaimer

        SchemeSense is an academic prototype. Its results are not
        official government decisions. Users must verify eligibility
        and scheme rules with the relevant government authorities.

        **Developed as an AI-based academic project.**
        """)

print("SchemeSense dashboard created successfully!")
print("Total tabs: 5")
print("Ready to launch!")


SchemeSense dashboard created successfully!
Total tabs: 5
Ready to launch!


In [ ]:
import gradio as gr

print("Gradio version:", gr.__version__)
print("Dashboard exists:", "professional_dashboard" in globals())
print("Theme exists:", "professional_theme" in globals())

Gradio version: 6.31.0
Dashboard exists: False
Theme exists: False


In [ ]:
import gradio as gr

professional_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
    neutral_hue="slate"
)

print("Professional theme restored successfully!")

Professional theme restored successfully!


In [ ]:
required_items = [
    "generate_multi_scheme_answer",
    "classify_pm_kisan",
    "documents_df",
    "professional_theme"
]

for item in required_items:
    print(item, ":", "Available" if item in globals() else "Missing")

generate_multi_scheme_answer : Missing
classify_pm_kisan : Missing
documents_df : Missing
professional_theme : Available


In [ ]:
required_items = [
    "documents",
    "all_chunks",
    "embeddings",
    "index",
    "client",
    "MODEL_NAME"
]

for item in required_items:
    print(item, ":", "Available" if item in globals() else "Missing")

documents : Missing
all_chunks : Missing
embeddings : Missing
index : Missing
client : Missing
MODEL_NAME : Missing


In [ ]:
!pip install -q openai pypdf faiss-cpu gradio numpy pandas

import numpy as np
import pandas as pd
import faiss
import gradio as gr
import io

from pypdf import PdfReader
from openai import OpenAI
from google.colab import files, userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    raise ValueError("NEXUS_API_KEY is missing from Colab Secrets.")

client = OpenAI(
    api_key=NEXUS_API_KEY.strip(),
    base_url="https://nexusapi.navigatelabs.ai"
)

MODEL_NAME = "nova-micro"

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Reply only: SchemeSense connected"}
    ]
)

print("API connection successful!")
print(response.choices[0].message.content)

API connection successful!
SchemeSense connected


In [ ]:
from google.colab import files
from pypdf import PdfReader
import io

documents = []

print("Upload your PM-KISAN sample PDF:")
uploaded_files = files.upload()

for filename, file_data in uploaded_files.items():
    reader = PdfReader(io.BytesIO(file_data))

    for page_number, page in enumerate(reader.pages, start=1):
        documents.append({
            "source": filename,
            "scheme": "PM-KISAN",
            "document_type": "sample",
            "page": page_number,
            "text": page.extract_text() or ""
        })

print("\nUpload your PMAY-G guideline PDF:")
uploaded_pmay = files.upload()

for filename, file_data in uploaded_pmay.items():
    reader = PdfReader(io.BytesIO(file_data))

    for page_number, page in enumerate(reader.pages, start=1):
        documents.append({
            "source": filename,
            "scheme": "PMAY-G",
            "document_type": "official",
            "page": page_number,
            "text": page.extract_text() or ""
        })

print("\nPDF documents restored successfully!")
print("Total pages:", len(documents))
print("PM-KISAN pages:", sum(d["scheme"] == "PM-KISAN" for d in documents))
print("PMAY-G pages:", sum(d["scheme"] == "PMAY-G" for d in documents))

Upload your PM-KISAN sample PDF:


Saving SchemeSense_PM_KISAN_Sample_Reference (1).pdf to SchemeSense_PM_KISAN_Sample_Reference (1).pdf

Upload your PMAY-G guideline PDF:


Saving RevisedPM-KISANOperationalGuidelines(English).pdf to RevisedPM-KISANOperationalGuidelines(English).pdf

PDF documents restored successfully!
Total pages: 13
PM-KISAN pages: 1
PMAY-G pages: 12


In [ ]:
# Correct the metadata of the uploaded PM-KISAN guidelines
for doc in documents:
    if "RevisedPM-KISAN" in doc["source"]:
        doc["scheme"] = "PM-KISAN"
        doc["document_type"] = "official"

# Upload the actual PMAY-G guidelines
print("Upload your PMAY-G guideline PDF:")
uploaded_pmay = files.upload()

for filename, file_data in uploaded_pmay.items():
    reader = PdfReader(io.BytesIO(file_data))

    for page_number, page in enumerate(reader.pages, start=1):
        documents.append({
            "source": filename,
            "scheme": "PMAY-G",
            "document_type": "official",
            "page": page_number,
            "text": page.extract_text() or ""
        })

print("\nDocuments corrected and restored!")
print("Total pages:", len(documents))
print("PM-KISAN pages:", sum(d["scheme"] == "PM-KISAN" for d in documents))
print("PMAY-G pages:", sum(d["scheme"] == "PMAY-G" for d in documents))

Upload your PMAY-G guideline PDF:


Saving 20250311250605284.pdf to 20250311250605284.pdf

Documents corrected and restored!
Total pages: 35
PM-KISAN pages: 13
PMAY-G pages: 22


In [ ]:
def create_chunks(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []

    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

    return chunks


all_chunks = []

for doc in documents:
    for chunk in create_chunks(doc["text"]):
        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "scheme": doc["scheme"],
            "document_type": doc["document_type"]
        })


print("Knowledge chunks created successfully!")
print("Total document pages:", len(documents))
print("Total chunks:", len(all_chunks))

for scheme in ["PM-KISAN", "PMAY-G"]:
    count = sum(
        1 for chunk in all_chunks
        if chunk["scheme"] == scheme
    )
    print(scheme, "chunks:", count)

Knowledge chunks created successfully!
Total document pages: 35
Total chunks: 84
PM-KISAN chunks: 44
PMAY-G chunks: 40


In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = []

for start in range(0, len(texts), 50):
    batch = texts[start:start + 50]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    all_embeddings.extend([
        item.embedding for item in response.data
    ])

embeddings = np.array(all_embeddings, dtype="float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embeddings generated successfully!")
print("Total vectors:", index.ntotal)
print("Dimensions:", embeddings.shape[1])

Embeddings generated successfully!
Total vectors: 84
Dimensions: 1536


In [ ]:
def search_selected_scheme(question, selected_scheme, top_k=5):

    selected_indices = [
        i for i, chunk in enumerate(all_chunks)
        if chunk["scheme"] == selected_scheme
    ]

    if not selected_indices:
        return []

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scheme_vectors = embeddings[selected_indices]

    temp_index = faiss.IndexFlatIP(
        scheme_vectors.shape[1]
    )

    temp_index.add(scheme_vectors)

    k = min(top_k, len(selected_indices))

    scores, positions = temp_index.search(
        query_vector, k
    )

    results = []

    for position, score in zip(positions[0], scores[0]):

        if position >= 0:
            original_index = selected_indices[position]

            result = all_chunks[original_index].copy()
            result["similarity_score"] = float(score)

            results.append(result)

    return results


print("Document search function restored successfully!")

Document search function restored successfully!


In [ ]:
def generate_multi_scheme_answer(question, selected_scheme):

    results = search_selected_scheme(
        question,
        selected_scheme,
        top_k=5
    )

    if not results:
        return "No relevant information found in the uploaded documents."

    context = ""

    for i, result in enumerate(results, start=1):
        context += f"""
Passage {i}
Scheme: {result['scheme']}
Source: {result['source']}
Page: {result['page']}
Document type: {result['document_type']}
Text: {result['text']}
"""

    prompt = f"""
You are SchemeSense, a government scheme information assistant.

Selected scheme: {selected_scheme}

Answer using ONLY the retrieved document context.

Rules:
1. Use simple English.
2. Do not invent facts or eligibility rules.
3. If information is missing, say so clearly.
4. Mention the source filename and page number.
5. Do not claim final government eligibility or approval.
6. Clearly identify sample documents as samples, not official guidelines.
7. Distinguish official guidelines from sample references.

Retrieved context:
{context}

User question:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a careful government scheme information assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


print("SchemeSense AI chatbot restored successfully!")

SchemeSense AI chatbot restored successfully!


In [ ]:

def classify_pm_kisan(details):

    exclusion_reasons = []
    missing_details = []

    required_fields = [
        "Name", "State", "Age", "Farmer family members",
        "Agricultural land ownership", "Land record available",
        "Institutional landholding", "Constitutional or elected position",
        "Government employment", "Paid income tax last assessment year",
        "Registered practicing professional", "NRI status",
        "Monthly pension"
    ]

    for field in required_fields:
        if details.get(field) in [None, ""]:
            missing_details.append(f"{field} is missing.")

    yes_no_fields = required_fields[4:12]

    for field in yes_no_fields:
        answer = str(details.get(field, "")).strip().lower()
        if answer not in ["yes", "no"]:
            if f"{field} is missing." not in missing_details:
                missing_details.append(f"Please confirm: {field}.")

    if str(details.get("Agricultural land ownership", "")).lower() == "no":
        exclusion_reasons.append(
            "Agricultural land ownership requirement may not be met."
        )

    exclusion_fields = [
        "Institutional landholding",
        "Constitutional or elected position",
        "Government employment",
        "Paid income tax last assessment year",
        "Registered practicing professional",
        "NRI status"
    ]

    for field in exclusion_fields:
        if str(details.get(field, "")).strip().lower() == "yes":
            exclusion_reasons.append(
                f"Possible exclusion: {field}."
            )

    try:
        pension = float(details.get("Monthly pension"))

        if pension < 0:
            missing_details.append(
                "Monthly pension cannot be negative."
            )
        elif pension >= 10000:
            exclusion_reasons.append(
                "Monthly pension is ₹10,000 or more; "
                "official rules and exceptions must be checked."
            )

    except (ValueError, TypeError):
        if "Monthly pension is missing." not in missing_details:
            missing_details.append(
                "Enter a valid monthly pension amount."
            )

    if missing_details:
        status = "MORE INFORMATION REQUIRED"

    elif exclusion_reasons:
        status = "POSSIBLE EXCLUSION – OFFICIAL REVIEW REQUIRED"

    else:
        status = (
            "NO LISTED EXCLUSION DETECTED – "
            "OFFICIAL VERIFICATION REQUIRED"
        )

    return {
        "status": status,
        "exclusion_reasons": exclusion_reasons,
        "missing_details": missing_details
    }


print("PM-KISAN eligibility checker restored successfully!")


PM-KISAN eligibility checker restored successfully!


In [ ]:
# Create unique document records

grouped_docs = {}

for doc in documents:
    key = (
        doc["source"],
        doc["scheme"],
        doc["document_type"]
    )

    if key not in grouped_docs:
        grouped_docs[key] = set()

    grouped_docs[key].add(doc["page"])


document_info = []

for (source, scheme, doc_type), pages in grouped_docs.items():
    document_info.append({
        "File Name": source,
        "Scheme": scheme,
        "Document Type": doc_type,
        "Pages": len(pages)
    })


documents_df = pd.DataFrame(document_info)


# Restore professional theme

professional_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo",
    neutral_hue="slate"
)


print("Documents table restored!")
display(documents_df)

print("\nProfessional theme restored!")
print("Total unique documents:", len(documents_df))
print("Total knowledge chunks:", len(all_chunks))

Documents table restored!


,File Name,Scheme,Document Type,Pages
0,SchemeSense_PM_KISAN_Sample_Reference (1).pdf,PM-KISAN,sample,1
1,RevisedPM-KISANOperationalGuidelines(English).pdf,PM-KISAN,official,12
2,20250311250605284.pdf,PMAY-G,official,22



Professional theme restored!
Total unique documents: 3
Total knowledge chunks: 84


In [ ]:

# ============================================
# SCHEMESENSE - FINAL PROFESSIONAL DASHBOARD
# ============================================

def chatbot_response(message, history, scheme):
    return generate_multi_scheme_answer(message, scheme)


def eligibility_result(
    name, state, age, family_members,
    land_ownership, land_record, institutional_land,
    constitutional_position, government_job,
    income_tax, professional, nri, pension
):
    details = {
        "Name": name,
        "State": state,
        "Age": age,
        "Farmer family members": family_members,
        "Agricultural land ownership": land_ownership,
        "Land record available": land_record,
        "Institutional landholding": institutional_land,
        "Constitutional or elected position": constitutional_position,
        "Government employment": government_job,
        "Paid income tax last assessment year": income_tax,
        "Registered practicing professional": professional,
        "NRI status": nri,
        "Monthly pension": pension
    }

    result = classify_pm_kisan(details)

    output = f"""
## Eligibility Screening Result

**Applicant:** {name or "Not provided"}

**State:** {state or "Not provided"}

### Status: {result['status']}
"""

    if result["missing_details"]:
        output += "\n### Information Required\n"
        for item in result["missing_details"]:
            output += f"- {item}\n"

    if result["exclusion_reasons"]:
        output += "\n### Possible Exclusion Reasons\n"
        for item in result["exclusion_reasons"]:
            output += f"- {item}\n"

    output += """
---
**Disclaimer:** This is a preliminary AI screening result,
not an official government eligibility decision or approval.
Please verify the information with PM-KISAN authorities.
"""
    return output


with gr.Blocks(
    title="SchemeSense | Smart Government Portal"
) as professional_dashboard:

    # HEADER
    gr.Markdown("""
    # 🏛️ SCHEMESENSE
    ### AI-Based Government Scheme Information & Eligibility Assistant

    Discover government schemes, ask questions and check
    preliminary eligibility using uploaded scheme documents.

    ---
    """)

    # TAB 1 - HOME
    with gr.Tab("🏠 Home"):

        gr.Markdown("""
        ## Welcome to SchemeSense

        Your intelligent assistant for understanding government schemes.

        ### Dashboard Overview
        """)

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 📄 3\nUploaded Documents")
            with gr.Column():
                gr.Markdown("### 📚 84\nKnowledge Chunks")
            with gr.Column():
                gr.Markdown("### 🤖 AI + RAG\nTechnology")

        gr.Markdown("""
        ---

        ### Our Services

        **💬 AI Chatbot**

        Ask questions about PM-KISAN and PMAY-G.

        **📝 Eligibility Checker**

        Check preliminary PM-KISAN eligibility.

        **📂 Documents**

        View uploaded government scheme documents.

        **ℹ️ About**

        Learn about the SchemeSense project.
        """)

    # TAB 2 - AI CHATBOT
    with gr.Tab("💬 AI Chatbot"):

        gr.Markdown("""
        ## SchemeSense AI Chatbot

        Ask questions about government schemes using document-based AI.
        """)

        chatbot_scheme = gr.Dropdown(
            choices=["PM-KISAN", "PMAY-G"],
            value="PM-KISAN",
            label="Select Government Scheme"
        )

        gr.ChatInterface(
            fn=chatbot_response,
            additional_inputs=[chatbot_scheme],
            examples=[
                ["What is PM-KISAN?", "PM-KISAN"],
                ["What are the benefits of PM-KISAN?", "PM-KISAN"],
                ["What is PMAY-G?", "PMAY-G"],
                ["Explain PMAY-G eligibility.", "PMAY-G"]
            ]
        )

    # TAB 3 - ELIGIBILITY CHECKER
    with gr.Tab("📝 Eligibility Checker"):

        gr.Markdown("""
        ## PM-KISAN Eligibility Checker

        Enter applicant details for preliminary screening.
        Use fictional information for project demonstrations.
        """)

        with gr.Row():
            name = gr.Textbox(label="Applicant Name")
            state = gr.Textbox(label="State")

        with gr.Row():
            age = gr.Textbox(label="Age")
            family_members = gr.Textbox(
                label="Farmer family members"
            )

        land_ownership = gr.Dropdown(
            ["Yes", "No"],
            label="Agricultural land ownership"
        )

        land_record = gr.Dropdown(
            ["Yes", "No"],
            label="Land record available"
        )

        institutional_land = gr.Dropdown(
            ["Yes", "No"],
            label="Institutional landholding"
        )

        constitutional_position = gr.Dropdown(
            ["Yes", "No"],
            label="Constitutional or elected position"
        )

        government_job = gr.Dropdown(
            ["Yes", "No"],
            label="Government employment"
        )

        income_tax = gr.Dropdown(
            ["Yes", "No"],
            label="Paid income tax last assessment year"
        )

        professional = gr.Dropdown(
            ["Yes", "No"],
            label="Registered practicing professional"
        )

        nri = gr.Dropdown(
            ["Yes", "No"],
            label="NRI status"
        )

        pension = gr.Textbox(
            label="Monthly pension amount (₹)"
        )

        check_button = gr.Button(
            "Check Eligibility",
            variant="primary"
        )

        eligibility_output = gr.Markdown()

        check_button.click(
            fn=eligibility_result,
            inputs=[
                name, state, age, family_members,
                land_ownership, land_record,
                institutional_land, constitutional_position,
                government_job, income_tax,
                professional, nri, pension
            ],
            outputs=eligibility_output
        )

    # TAB 4 - DOCUMENTS
    with gr.Tab("📂 Documents"):

        gr.Markdown("""
        ## Uploaded Government Scheme Documents

        These documents are used as the SchemeSense knowledge base.
        """)

        gr.Dataframe(
            value=documents_df,
            interactive=False,
            label="Uploaded Documents"
        )

        gr.Markdown("""
        ### Document Processing Pipeline

        1. PDF Text Extraction
        2. Text Chunking
        3. Embedding Generation
        4. FAISS Vector Database
        5. RAG-Based Answer Generation
        """)

    # TAB 5 - ABOUT
    with gr.Tab("ℹ️ About"):

        gr.Markdown("""
        ## About SchemeSense

        SchemeSense is an AI-based government scheme information
        and preliminary eligibility assistant.

        ### Project Objectives

        - Simplify government scheme information.
        - Answer questions using uploaded documents.
        - Support preliminary eligibility screening.
        - Provide source-based AI responses.

        ### Technologies Used

        | Technology | Purpose |
        |---|---|
        | Python | Backend |
        | Gradio | User Interface |
        | FAISS | Vector Database |
        | Embeddings | Text Representation |
        | RAG | Document Retrieval |
        | Nexus API | AI Model |

        ### Disclaimer

        SchemeSense is an academic prototype.
        Its results are not official government decisions.
        Always verify eligibility with the relevant authorities.

        **Developed as an AI academic project.**
        """)


print("SchemeSense dashboard created successfully!")
print("Total tabs: 5")
print("Documents:", len(documents_df))
print("Knowledge chunks:", len(all_chunks))
print("Ready to launch!")


SchemeSense dashboard created successfully!
Total tabs: 5
Documents: 3
Knowledge chunks: 84
Ready to launch!


In [ ]:
import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.31.0


In [ ]:
print("PDF chunks:", "all_chunks" in globals())
print("FAISS index:", "index" in globals())
print("Documents:", "documents_df" in globals())
print("Theme:", "professional_theme" in globals())
print("Chat function:", "generate_multi_scheme_answer" in globals())

PDF chunks: False
FAISS index: False
Documents: False
Theme: False
Chat function: False


In [ ]:
import os

print("Available PDF files:")

for file in os.listdir("/content"):
    if file.endswith(".pdf"):
        print(file)

Available PDF files:


In [ ]:
from google.colab import files

uploaded_files = files.upload()

print("Uploaded files:")
for filename in uploaded_files:
    print(filename)

Saving 20250311250605284.pdf to 20250311250605284.pdf
Uploaded files:
20250311250605284.pdf


In [ ]:
from google.colab import files
uploaded_files = files.upload()

Saving SchemeSense_PM_KISAN_Sample_Reference (1).pdf to SchemeSense_PM_KISAN_Sample_Reference (1).pdf


In [ ]:
from google.colab import files

uploaded_files = files.upload()

Saving SchemeSense_PM_KISAN_Sample_Reference.pdf to SchemeSense_PM_KISAN_Sample_Reference.pdf


In [ ]:
import os

for f in os.listdir("/content"):
    if f.lower().endswith(".pdf"):
        print(f)

SchemeSense_PM_KISAN_Sample_Reference.pdf
RevisedPM-KISANOperationalGuidelines(English).pdf
SchemeSense_PM_KISAN_Sample_Reference (1).pdf
20250311250605284.pdf


In [ ]:
!pip install -q pypdf faiss-cpu sentence-transformers gradio pandas numpy openai

In [ ]:
import os
from google.colab import userdata

try:
    api_key = userdata.get("NEXUS_API_KEY")
    print("Nexus API key:", "Available" if api_key else "Not found")
except Exception as e:
    print("API key error:", e)

Nexus API key: Available


In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("NEXUS_API_KEY"),
    base_url="https://nexusapi.navigatelabs.ai/v1"
)

print("Nexus API connected successfully!")

Nexus API connected successfully!


In [ ]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("NEXUS_API_KEY").strip()

client = OpenAI(
    api_key=api_key,
    base_url="https://nexusapi.navigatelabs.ai/v1"
)

print("Nexus API connected successfully!")

Nexus API connected successfully!


In [ ]:
response = client.chat.completions.create(
    model="nova-micro",
    messages=[
        {"role": "user", "content": "Hello, I am SchemeSense AI."}
    ]
)

print(response.choices[0].message.content)

Hello, SchemeSense AI! It's great to have you here. Whether you're interested in discussing programming, algorithms, data structures, or anything else related to computer science and software development, I'm here to help. How can I assist you today? If you have any specific questions or topics you'd like to explore, just let me know!


In [ ]:
from pypdf import PdfReader
import os

pdf_files = [
    "SchemeSense_PM_KISAN_Sample_Reference.pdf",
    "RevisedPM-KISANOperationalGuidelines(English).pdf",
    "20250311250605284.pdf"
]

documents = []

for file in pdf_files:
    reader = PdfReader(file)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    documents.append({
        "filename": file,
        "text": text
    })

print("Total documents:", len(documents))

for doc in documents:
    print(doc["filename"], "Characters:", len(doc["text"]))

Total documents: 3
SchemeSense_PM_KISAN_Sample_Reference.pdf Characters: 2420
RevisedPM-KISANOperationalGuidelines(English).pdf Characters: 28141
20250311250605284.pdf Characters: 27264


In [ ]:
all_chunks = []

for doc in documents:
    text = doc["text"]
    filename = doc["filename"]

    chunk_size = 1000
    overlap = 200

    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]

        if chunk.strip():
            all_chunks.append({
                "text": chunk,
                "filename": filename
            })

print("Total chunks:", len(all_chunks))

Total chunks: 75


In [ ]:
from openai import OpenAI
import numpy as np

texts = [chunk["text"] for chunk in all_chunks]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = np.array(
    [item.embedding for item in response.data],
    dtype="float32"
)

print("Embeddings generated successfully!")
print("Total vectors:", len(embeddings))
print("Dimensions:", embeddings.shape[1])

Embeddings generated successfully!
Total vectors: 75
Dimensions: 1536


In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Total vectors:", index.ntotal)

FAISS index created successfully!
Total vectors: 75


In [ ]:
def search_documents(question, top_k=5):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_vector = np.array(
        [response.data[0].embedding],
        dtype="float32"
    )

    distances, indices = index.search(query_vector, top_k)

    results = []

    for i in indices[0]:
        results.append(all_chunks[i])

    return results

print("Document search function created!")

Document search function created!


In [ ]:
results = search_documents(
    "What are the eligibility rules for PM-KISAN?"
)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("File:", result["filename"])
    print(result["text"][:300])


--- Result 1 ---
File: RevisedPM-KISANOperationalGuidelines(English).pdf
 employees) 
 
iv) All superannuated / retired pensioners whose monthly pension is Rs.10,000/- or 
more (Excluding Multi Tasking Staff / Class IV / Group D employees) 
 
v) All Persons who paid Income Tax in last assessment year. 
 
vi) Professionals like Doctors, Engineers, Lawyers, Chartered Accou

--- Result 2 ---
File: SchemeSense_PM_KISAN_Sample_Reference.pdf
SchemeSense – PM-KISAN Sample Reference
 PDF test document for text extraction, chunking, embeddings and retrieval
Important: This is a student-created summary for testing the SchemeSense pipeline. It is not an official Government
of India notification and must not be used as the final authority for

--- Result 3 ---
File: RevisedPM-KISANOperationalGuidelines(English).pdf
1 
 
 
 
 
 
PRADHAN MANTRI KISAN SAMMAN NIDHI SCHEME 
 
 
(PM-KISAN SCHEME) 
 
 
 
 
 
OPERATIONAL GUIDELINES  
 
(REVISED AS ON 29.03.2020) 
 
 
 
 
 
 
MINISTRY OF AGRICULTURE & FAR

In [ ]:
def generate_answer(question):
    results = search_documents(question, top_k=5)

    context = "\n\n".join(
        [r["text"] for r in results]
    )

    response = client.chat.completions.create(
        model="nova-micro",
        messages=[
            {
                "role": "system",
                "content": "You are SchemeSense, a government scheme information assistant. Answer using the provided document context. If information is missing, say so. Do not invent eligibility rules."
            },
            {
                "role": "user",
                "content": f"""
                Question: {question}

                Document Context:
                {context}

                Give a simple and clear answer.
                """
            }
        ]
    )

    return response.choices[0].message.content

print("AI answer function created successfully!")

AI answer function created successfully!


In [ ]:
import gradio as gr
import pandas as pd

documents_df = pd.DataFrame([
    {
        "Document": doc["filename"],
        "Characters": len(doc["text"]),
        "Status": "Processed"
    }
    for doc in documents
])

def chatbot_response(message, history):
    try:
        return generate_answer(message)
    except Exception as e:
        return f"Error: {str(e)}"

def check_eligibility(name, land, government_job, income):
    if not name.strip():
        return "Please enter applicant name."

    if land == "No":
        return "Preliminary result: Landholding condition may not be satisfied. Verify official rules."

    if government_job == "Yes":
        return "Possible exclusion condition. Official verification is required."

    return (
        f"Applicant: {name}\n"
        "Preliminary result: Potentially eligible based on the limited details provided.\n"
        "This is not an official eligibility decision."
    )

professional_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="indigo"
)

with gr.Blocks(title="SchemeSense AI Dashboard") as professional_dashboard:

    gr.Markdown("""
    # 🏛️ SCHEMESENSE
    ### AI-Based Government Scheme Information & Eligibility Assistant

    Discover government schemes, ask questions and check preliminary eligibility.

    ---
    """)

    with gr.Tabs():

        with gr.Tab("🏠 Home"):
            gr.Markdown("""
            ## Welcome to SchemeSense!

            Your AI-powered government scheme assistant.

            ### Key Features
            - AI-powered chatbot
            - Preliminary eligibility checker
            - Document-based information
            - Government scheme document library

            ### Supported Schemes
            - PM-KISAN
            - PMAY-G

            ### Project Workflow
            PDF → Text Extraction → Chunking → Embeddings → FAISS → AI Answer
            """)

        with gr.Tab("🤖 AI Chatbot"):
            gr.Markdown("## Ask SchemeSense")

            gr.ChatInterface(
                fn=chatbot_response,
                examples=[
                    "What is PM-KISAN?",
                    "What are PM-KISAN eligibility conditions?",
                    "What is PMAY-G?",
                    "What documents are required?"
                ]
            )

        with gr.Tab("✅ Eligibility Checker"):
            gr.Markdown("## Preliminary Eligibility Checker")

            applicant_name = gr.Textbox(label="Applicant Name")

            land = gr.Radio(
                ["Yes", "No"],
                label="Does the applicant's family hold cultivable land?"
            )

            government_job = gr.Radio(
                ["Yes", "No"],
                label="Does any family member hold a government job?"
            )

            income = gr.Dropdown(
                ["Below ₹2 lakh", "₹2–5 lakh", "Above ₹5 lakh"],
                label="Annual Family Income (Demo Field)"
            )

            check_button = gr.Button(
                "Check Eligibility",
                variant="primary"
            )

            eligibility_output = gr.Textbox(
                label="Screening Result",
                lines=5
            )

            check_button.click(
                check_eligibility,
                inputs=[applicant_name, land, government_job, income],
                outputs=eligibility_output
            )

        with gr.Tab("📚 Documents"):
            gr.Markdown("## Uploaded Scheme Documents")

            gr.Dataframe(
                value=documents_df,
                interactive=False
            )

            gr.Markdown("""
            - Total PDFs: 3
            - Total chunks: 75
            - Embedding dimensions: 1536
            - Vector database: FAISS
            """)

        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            ## About SchemeSense

            SchemeSense is an academic RAG-based government scheme assistant.

            ### Technologies
            - Python
            - Gradio
            - Nexus API
            - Nova Micro
            - Text Embeddings
            - FAISS
            - PyPDF

            **Note:** Results are preliminary and must be verified
            against current official government guidelines.
            """)

    gr.Markdown("---\n© 2026 SchemeSense | Academic Project")

print("SchemeSense Dashboard created successfully!")

SchemeSense Dashboard created successfully!


In [ ]:
professional_dashboard.launch(
    share=True,
    debug=True,
    theme=professional_theme,
    css="""
    .gradio-container {
        max-width: 1200px !important;
        margin: auto !important;
    }
    """
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://320dcc5e9c1bac035e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
